# CS781 Phase 2 — Legal Named Entity Recognition (L-NER)

Legal NER on Indian judgments: predict exact character spans for 12 entity types, starting from a Phase-1 legal language model.

Pipeline:

`document → tokenizer + exact character offsets → Phase-1 backbone → token NER heads (BIO + start/end boundary) → character spans → prediction.json`

**Backbones used in this notebook**

| Backbone | Phase-1 model | Adaptation methods run here |
|---|---|---|
| `decoder` | Decoder-only causal Transformer (8 layers, d=640) | full fine-tuning, partial fine-tuning (last 2 blocks), LORMA, knowledge distillation (teacher: full fine-tuned decoder) |
| `ssm` | Diagonal SSM / S4D-inspired LM (10 layers, d=700) | full fine-tuning |

Select the experiment in section 10 (`exp_cfg`). Section 14 (knowledge distillation) uses the full fine-tuned decoder as the teacher, so run the `decoder` / `full` configuration first.

> **Not part of this phase:** the encoder-decoder and CNN + attention Phase-1 models were not adapted for L-NER in this notebook; they are left as future work.


## 0. Workflow and Data Setup

Order followed in this notebook:

`inspect data → exact offset mapping → windowing → backbone → training with validation → span decoding → threshold tuning → knowledge distillation → test inference → submission`


In [ ]:
import os, json, random, math, time, zipfile
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


Device: cuda:0


In [ ]:
# ============================================================
# HUGGING FACE SETUP
# ============================================================

!pip install -q huggingface_hub safetensors
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("hf_token_rp")

from pathlib import Path
from huggingface_hub import login, hf_hub_download, snapshot_download

HF_TOKEN = secret_value_0

login(
    token=HF_TOKEN,
    add_to_git_credential=False
)

# ============================================================
# PHASE 2 DATASET FROM HUGGING FACE
# ============================================================

DATA_REPO = "Exploration-Lab/CS-781-Capstone"

def get_data_file(filename):
    return Path(
        hf_hub_download(
            repo_id=DATA_REPO,
            filename=filename,
            repo_type="dataset",
            token=HF_TOKEN
        )
    )


FOLD1_TEXT = get_data_file(
    "Phase_2_LNER/Train/fold1-text.json"
)

FOLD1_GOLD = get_data_file(
    "Phase_2_LNER/Train/fold1-gold.json"
)

FOLD2_TEXT = get_data_file(
    "Phase_2_LNER/Train/fold2-text.json"
)

FOLD2_GOLD = get_data_file(
    "Phase_2_LNER/Train/fold2-gold.json"
)

LABELS_FILE = get_data_file(
    "Phase_2_LNER/Train/labels.txt"
)

TEST_TEXT = get_data_file(
    "Phase_2_LNER/Test/fold3-text.json"
)


print("Fold 1 text :", FOLD1_TEXT)
print("Fold 1 gold :", FOLD1_GOLD)
print("Fold 2 text :", FOLD2_TEXT)
print("Fold 2 gold :", FOLD2_GOLD)
print("Labels      :", LABELS_FILE)
print("Test        :", TEST_TEXT)

Fold 1 text : /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_2_LNER/Train/fold1-text.json
Fold 1 gold : /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_2_LNER/Train/fold1-gold.json
Fold 2 text : /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_2_LNER/Train/fold2-text.json
Fold 2 gold : /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_2_LNER/Train/fold2-gold.json
Labels      : /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_2_LNER/Train/labels.txt
Test        : /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_2_

## 1. Load Phase-2 data

Do **not** normalize, strip, or collapse whitespace in documents used for character offsets.


In [ ]:
def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def load_labels(path):
    with open(path, "r", encoding="utf-8") as f:
        return [x.strip() for x in f if x.strip()]

fold1_text = load_json(FOLD1_TEXT)
fold1_gold = load_json(FOLD1_GOLD)
fold2_text = load_json(FOLD2_TEXT)
fold2_gold = load_json(FOLD2_GOLD)
test_text = load_json(TEST_TEXT)
entity_labels = load_labels(LABELS_FILE)

print("fold1:", len(fold1_text))
print("fold2:", len(fold2_text))
print("test :", len(test_text))
print("labels:", entity_labels)


fold1: 35
fold2: 35
test : 35
labels: ['APP', 'RESP', 'A.COUNSEL', 'R.COUNSEL', 'JUDGE', 'WIT', 'AUTH', 'COURT', 'STAT', 'PREC', 'DATE', 'CASENO']


In [ ]:
def preview_doc(text_map, gold_map, doc_id=None, max_chars=1200):
    if doc_id is None:
        doc_id = next(iter(text_map))
    text = text_map[doc_id]
    anns = gold_map.get(doc_id, [])
    print("DOC ID:", doc_id)
    print("\nTEXT:\n", repr(text[:max_chars]))
    print("\nRAW ANNOTATIONS:")
    for ann in anns[:30]:
        print(ann)

preview_doc(fold1_text, fold1_gold,doc_id="115651329")


DOC ID: 115651329

TEXT:
 'REPORTABLE IN THE SUPREME COURT OF INDIA CRIMINAL APPELLATE JURISDICTION CRIMINAL APPEAL NO. 92/2015 JAGE RAM & ORS. ..Appellants Versus STATE OF HARYANA ..Respondent J U D G M E N T R. BANUMATHI, J. This appeal is preferred against the judgment dated 19.8.2011 passed by the High Court of Punjab and Haryana in Criminal Appeal No.181 SB of 2000, whereby the High Court partly allowed the appeal filed by the appellants thereby confirming the conviction of the appellants with certain modifications. 2. Briefly stated, case of the prosecution is that on the fateful day i.e. 18.11.1994, at about 8.00 A.M. in the morning the complainant Jagdish (PW-5) along with his two sons namely Sukhbir and Mange Ram (PW-6) were busy in cutting pullas (reeds) from the dola of their field. At that time, Jage Ram (A-1) and his sons Rajbir Singh @ Raju (A-2), Rakesh (A-3) and Madan (A-4) armed with jaily, pharsi and lathis respectively, entered the land where the complainant was work

## 2. Normalize gold annotations

After inspecting the actual JSON, make **one function** that converts every annotation to:

`(start_char, end_char, label)`

First verify whether the dataset uses an exclusive or inclusive end offset.


In [ ]:
def normalize_gold(raw_annotations):
    spans = []

    for ann in raw_annotations:
        start = int(ann[0])
        end = int(ann[1])
        label = ann[2]

        spans.append([start, end, label])

    return spans

In [ ]:
def validate_gold_spans(text, spans, valid_labels):
    errors = []
    valid_labels = set(valid_labels)

    for i, row in enumerate(spans):
        if len(row) < 3:
            errors.append((i, "bad row", row))
            continue

        start, end, label = row[:3]

        if not isinstance(start, int) or not isinstance(end, int):
            errors.append((i, "non-integer offsets", row))
            continue

        if not (0 <= start < end <= len(text)):
            errors.append((i, "out of range", row, len(text)))

        if label not in valid_labels:
            errors.append((i, "invalid label", row))

    return errors


In [ ]:
doc_id="115651329"
# anns = gold_map.get(doc_id, [])

spans=normalize_gold(fold1_gold.get(doc_id,[]))
errors = validate_gold_spans(
    fold1_text[doc_id],
    spans,
    entity_labels
)

print(errors[:20])
print("num errors:", len(errors))

[]
num errors: 0


## 3. Phase-1 ByteLevelBPE tokenizer with character offsets

The same `ByteLevelBPE` tokenizer trained in Phase 1 is loaded below.

### Phase-2 addition

`ids, offsets = encode_with_offsets(tokenizer, text)`

with:

- `len(ids) == len(offsets)`
- `offsets[i] = (start_char, end_char)`
- offsets refer to the **original document string**
- no whitespace normalization before offset calculation
- no decoding-and-searching to recover positions


In [ ]:
#BPE tokenizer

def train_tokenizer(train_text, vocab_size):
    tokenizer = ByteLevelBPE(vocab_size=vocab_size)
    tokenizer.train([train_text])
    return tokenizer


import json
from collections import Counter, defaultdict
from tqdm.auto import tqdm


BPE_SPECIALS = [
    "<pad>",
    "<unk>",
    "<s>",
    "</s>"
]

# End-of-word marker.
# UTF-8 bytes are integers 0..255, so 256 cannot collide.
EOW = 256


class ByteLevelBPE:
    """
    Deterministic byte-level BPE tokenizer.

    - Initial units are UTF-8 bytes.
    - EOW marks the end of each word.
    - Case is preserved.
    - BPE never crosses word boundaries.
    - Ties are broken lexicographically.
    - Four special tokens are fixed at the beginning.
    """

    def __init__(self, vocab_size=5000):

        self.vocab_size = vocab_size
        self.merge_ranks = {}
        self.specials = list(BPE_SPECIALS)

        # List of learned merges.
        #
        # Each element is:
        #     (left_symbol, right_symbol)
        #
        self.merges = []

        # Vocabulary.
        self.itos = []
        self.stoi = {}

        # Internal merge rules.
        #
        # (left, right) -> merged_symbol
        #
        self.merge_rules = {}

        # Final representations of words seen during training.
        self.word_cache = {}
        self.token_to_symbol = {}

    # --------------------------------------------------------
    # Convert a byte/symbol tuple into a deterministic string
    # --------------------------------------------------------
    @staticmethod
    def string_to_symbol(token):
        # Standalone end-of-word marker
        if token == "</w>":
            return (EOW,)
    
        # Merged symbol ending with EOW
        if token.endswith("</w>>"):
            hex_part = token[3:-5]
            byte_part = tuple(
                int(hex_part[i:i+2], 16)
                for i in range(0, len(hex_part), 2)
            )
            return byte_part + (EOW,)
    
        # Ordinary byte or merged byte symbol
        hex_part = token[3:-1]
        return tuple(
            int(hex_part[i:i+2], 16)
            for i in range(0, len(hex_part), 2)
        )
    @staticmethod
    def symbol_to_string(symbol):

        # EOW is always the last symbol when present.
        if symbol and symbol[-1] == EOW:

            byte_part = symbol[:-1]

            if len(byte_part) == 0:
                return "</w>"

            hex_part = "".join(
                f"{b:02x}"
                for b in byte_part
            )

            return f"<0x{hex_part}</w>>"

        hex_part = "".join(
            f"{b:02x}"
            for b in symbol
        )

        return f"<0x{hex_part}>"
    def _build_merge_ranks(self):
        self.merge_ranks = {
            pair: rank
            for rank, pair in enumerate(self.merge_rules)
        }
    # --------------------------------------------------------
    # Deterministic ordering key for symbols
    # --------------------------------------------------------

    @staticmethod
    def symbol_key(symbol):
        return tuple(symbol)

    # --------------------------------------------------------
    # Convert a word into initial UTF-8 byte symbols
    # --------------------------------------------------------

    @staticmethod
    def word_to_symbols(word):

        byte_values = list(
            word.encode("utf-8")
        )

        symbols = [
            (b,)
            for b in byte_values
        ]

        # Explicit word boundary.
        symbols.append((EOW,))

        return symbols

    # --------------------------------------------------------
    # Count all pairs once initially
    # --------------------------------------------------------

    def _initial_pair_counts(
        self,
        word_symbols,
        word_counts
    ):

        pair_counts = Counter()

        # pair_words tells us which words contain a pair.
        pair_words = defaultdict(set)

        for word, symbols in word_symbols.items():

            freq = word_counts[word]

            for i in range(len(symbols) - 1):

                pair = (
                    symbols[i],
                    symbols[i + 1]
                )

                pair_counts[pair] += freq
                pair_words[pair].add(word)

        return pair_counts, pair_words

    # --------------------------------------------------------
    # Merge one pair in one word
    # --------------------------------------------------------

    @staticmethod
    def _merge_word(symbols, pair):

        left, right = pair

        new_symbol = left + right

        result = []

        i = 0

        while i < len(symbols):

            if (
                i + 1 < len(symbols)
                and symbols[i] == left
                and symbols[i + 1] == right
            ):

                result.append(new_symbol)

                i += 2

            else:

                result.append(symbols[i])

                i += 1

        return result

    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------

    def train(self, texts):

        print("=" * 70)
        print("Training BYTE-LEVEL BPE")
        print("=" * 70)

        # ----------------------------------------------------
        # Count unique words.
        # ----------------------------------------------------

        word_counts = Counter()

        for text in texts:
            if not isinstance(text, str):
                continue

            for word in text.split():

                word_counts[word] += 1

        print(
            f"Unique words: {len(word_counts):,}"
        )

        # ----------------------------------------------------
        # Initial representation of each word.
        # ----------------------------------------------------

        word_symbols = {
            word: self.word_to_symbols(word)
            for word in word_counts
        }

        # ----------------------------------------------------
        # Initial vocabulary symbols.
        # ----------------------------------------------------

        initial_symbols = [
            (b,)
            for b in range(256)
        ]

        initial_symbols.append((EOW,))

        # Four specials + initial byte symbols.
        current_vocab_size = (
            len(BPE_SPECIALS)
            + len(initial_symbols)
        )

        print(
            f"Initial vocabulary size: "
            f"{current_vocab_size:,}"
        )

        if current_vocab_size > self.vocab_size:

            raise ValueError(
                "Initial byte vocabulary already exceeds "
                "the requested vocabulary size."
            )

        # ----------------------------------------------------
        # Count pairs once.
        # ----------------------------------------------------

        pair_counts, pair_words = \
            self._initial_pair_counts(
                word_symbols,
                word_counts
            )

        # ----------------------------------------------------
        # Main BPE loop.
        # ----------------------------------------------------

        merge_no = 0

        while current_vocab_size < self.vocab_size:

            if not pair_counts:
                break

            # Ignore pairs occurring only once.
            valid_pairs = [
                pair
                for pair, count in pair_counts.items()
                if count > 1
            ]

            if not valid_pairs:
                print(
                    "No pair occurs more than once. "
                    "Stopping early."
                )
                break

            # Highest frequency first.
            # Lexicographically smallest pair on ties.
            best_pair = min(
                valid_pairs,
                key=lambda pair: (
                    -pair_counts[pair],
                    self.symbol_key(pair[0]),
                    self.symbol_key(pair[1])
                )
            )

            best_count = pair_counts[best_pair]

            left, right = best_pair
            new_symbol = left + right

            # ------------------------------------------------
            # Record merge.
            # ------------------------------------------------

            self.merge_rules[best_pair] = new_symbol

            self.merges.append(
                (
                    self.symbol_to_string(left),
                    self.symbol_to_string(right)
                )
            )

            # ------------------------------------------------
            # Words affected by this merge.
            # ------------------------------------------------

            affected_words = list(
                pair_words.get(best_pair, set())
            )

            # Remove the selected pair's old contribution.
            del pair_counts[best_pair]

            pair_words.pop(
                best_pair,
                None
            )

            # ------------------------------------------------
            # Update each affected word.
            # ------------------------------------------------

            for word in affected_words:

                old_symbols = word_symbols[word]

                old_pairs = []

                for i in range(
                    len(old_symbols) - 1
                ):

                    old_pairs.append(
                        (
                            old_symbols[i],
                            old_symbols[i + 1]
                        )
                    )

                # Remove old pair contributions.
                freq = word_counts[word]

                for pair in old_pairs:

                    pair_counts[pair] -= freq

                    if pair_counts[pair] <= 0:
                        pair_counts.pop(
                            pair,
                            None
                        )

                    if pair in pair_words:
                        pair_words[pair].discard(word)

                        if not pair_words[pair]:
                            pair_words.pop(
                                pair,
                                None
                            )

                # Perform merge.
                new_symbols = self._merge_word(
                    old_symbols,
                    best_pair
                )

                word_symbols[word] = new_symbols

                # Add new pair contributions.
                new_pairs = []

                for i in range(
                    len(new_symbols) - 1
                ):

                    new_pairs.append(
                        (
                            new_symbols[i],
                            new_symbols[i + 1]
                        )
                    )

                for pair in new_pairs:

                    pair_counts[pair] += freq
                    pair_words[pair].add(word)

            current_vocab_size += 1
            merge_no += 1

            if merge_no % 250 == 0:

                print(
                    f"Merges: {merge_no:4d} | "
                    f"Vocab: {current_vocab_size:5d} | "
                    f"Best pair frequency: {best_count:,}"
                )

        # ----------------------------------------------------
        # Construct final vocabulary.
        #
        # First:
        #   <pad>, <unk>, <s>, </s>
        #
        # Then initial symbols.
        #
        # Then symbols introduced by merges in merge order.
        # ----------------------------------------------------

        merged_symbols = list(
            self.merge_rules.values()
        )

        self.itos = (
            list(BPE_SPECIALS)
            +
            [
                self.symbol_to_string(s)
                for s in initial_symbols
            ]
            +
            [
                self.symbol_to_string(s)
                for s in merged_symbols
            ]
        )

        self.token_to_symbol = {}

        for symbol in initial_symbols:
            token = self.symbol_to_string(symbol)
            self.token_to_symbol[token] = symbol

        for symbol in merged_symbols:
            token = self.symbol_to_string(symbol)
            self.token_to_symbol[token] = symbol

        # Remove duplicates while preserving order.
        final_itos = []
        seen = set()

        for token in self.itos:

            if token not in seen:

                seen.add(token)
                final_itos.append(token)

        self.itos = final_itos[
            :self.vocab_size
        ]

        self.stoi = {
            token: i
            for i, token in enumerate(self.itos)
        }

        # Save representations of training words.
        self.word_cache = word_symbols

        print(
            "\nTraining complete."
        )

        print(
            f"Final vocabulary size: "
            f"{len(self.itos):,}"
        )

        print(
            f"Number of merges: "
            f"{len(self.merges):,}"
        )
        self._build_merge_ranks()
        return self

    # --------------------------------------------------------
    # Tokenize a word
    # --------------------------------------------------------

    def tokenize_word(self, word):
        # Use cached representation for training words.
        if word in self.word_cache:
            symbols = self.word_cache[word]
        else:
            symbols = self.word_to_symbols(word)
    
            while len(symbols) > 1:
                # Find adjacent pairs that have learned merges.
                candidates = [
                    (self.merge_ranks[(symbols[i], symbols[i + 1])], i)
                    for i in range(len(symbols) - 1)
                    if (symbols[i], symbols[i + 1]) in self.merge_ranks
                ]
    
                if not candidates:
                    break
    
                # Apply the earliest learned merge.
                _, best_idx = min(candidates)
    
                pair = (symbols[best_idx], symbols[best_idx + 1])
                new_symbol = self.merge_rules[pair]
    
                # Merge all non-overlapping occurrences.
                symbols = self._merge_word(symbols, pair)
    
        return [
            self.symbol_to_string(symbol)
            for symbol in symbols
        ]

    # --------------------------------------------------------
    # Tokenize complete sentence
    # --------------------------------------------------------

    def tokenize(self, text):

        tokens = []

        for word in text.split():

            tokens.extend(
                self.tokenize_word(word)
            )

        return tokens

    def encode(self, text):
        tokens = self.tokenize(text)

        ids = []
        for token in tokens:
            if token in self.stoi:
                ids.append(self.stoi[token])
            else:
                # unknown token
                ids.append(self.stoi["<unk>"])

        return ids

    def decode(self, ids):
      byte_values = []

      for idx in ids:
          token = self.itos[idx]

          if token in BPE_SPECIALS:
              continue

          symbol = self.token_to_symbol[token]

          for b in symbol:
              if b == EOW:
                  byte_values.append(ord(" "))
              else:
                  byte_values.append(b)

      # Remove the final space
      if byte_values and byte_values[-1] == ord(" "):
          byte_values.pop()

      return bytes(byte_values).decode("utf-8")

    @classmethod
    def load(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
    
        tokenizer = cls(vocab_size=data["vocab_size"])
        tokenizer.merges = data["merges"]
        tokenizer.itos = data["itos"]
        tokenizer.stoi = {
            token: i for i, token in enumerate(tokenizer.itos)
        }
    
        # Initial symbols
        initial_symbols = [(b,) for b in range(256)]
        initial_symbols.append((EOW,))
    
        tokenizer.token_to_symbol = {
            tokenizer.symbol_to_string(symbol): symbol
            for symbol in initial_symbols
        }
    
        # Reconstruct merge rules in training order
        for left_str, right_str in tokenizer.merges:
            left = tokenizer.string_to_symbol(left_str)
            right = tokenizer.string_to_symbol(right_str)
            merged = left + right
    
            tokenizer.merge_rules[(left, right)] = merged
            merged_str = tokenizer.symbol_to_string(merged)
            tokenizer.token_to_symbol[merged_str] = merged
        tokenizer._build_merge_ranks()  
        return tokenizer

    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    def save(self, path):

        data = {
            "merges": self.merges,
            "itos": self.itos,
            "vocab_size": self.vocab_size
        }

        with open(
            path,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                data,
                f,
                ensure_ascii=False
            )

        print(
            f"Saved: {path}"
        )

# if __name__ == "__main__":
#     train_text = """
#     Harry went to Hogwarts.
#     Harry met Ron and Hermione.
#     Hogwarts was a magical school.
#     """

#     tokenizer = train_tokenizer(train_text, 5000)

#     print("Vocab size:", tokenizer.vocab_size)

#     text = "Harry went to Hogwarts."

#     ids = tokenizer.encode(text)
#     decoded = tokenizer.decode(ids)

#     print("Text:", text)
#     print("IDs:", ids)
#     print("Decoded:", decoded)
#     print("Match:", text == decoded)

# PAD_ID = tokenizer.stoi["<pad>"]
# UNK_ID = tokenizer.stoi["<unk>"]
# BOS_ID = tokenizer.stoi["<s>"]
# EOS_ID = tokenizer.stoi["</s>"]


In [ ]:
#load tokenizer

tokenizer = ByteLevelBPE.load(
    "/kaggle/input/datasets/sg123321/tokenizer1/legal_bpe.json"
)

print("Vocabulary size:", len(tokenizer.itos))
print("Number of merges:", len(tokenizer.merges))

PAD_ID = tokenizer.stoi["<pad>"]
UNK_ID = tokenizer.stoi["<unk>"]
BOS_ID = tokenizer.stoi["<s>"]
EOS_ID = tokenizer.stoi["</s>"]

Vocabulary size: 16000
Number of merges: 15739


In [ ]:
import re

def encode_with_offsets(tokenizer, text):
    ids = []
    offsets = []

    # Finds words exactly like text.split(), but preserves
    # their positions in the ORIGINAL document.
    for match in re.finditer(r"\S+", text):
        word = match.group()
        word_start = match.start()
        word_end = match.end()

        tokens = tokenizer.tokenize_word(word)

        # Map UTF-8 bytes of this word back to character positions.
        byte_to_char = []

        for char_idx, ch in enumerate(word):
            b = ch.encode("utf-8")

            for _ in b:
                byte_to_char.append(char_idx)

        byte_cursor = 0

        for token in tokens:
            token_id = tokenizer.stoi.get(
                token,
                tokenizer.stoi["<unk>"]
            )

            symbol = tokenizer.token_to_symbol[token]

            # EOW is not an actual character in the source text.
            real_bytes = [b for b in symbol if b != EOW]

            if len(real_bytes) == 0:
                # Standalone EOW token.
                start_char = word_end
                end_char = word_end

            else:
                byte_start = byte_cursor
                byte_end = byte_cursor + len(real_bytes)

                local_start = byte_to_char[byte_start]
                local_end = byte_to_char[byte_end - 1] + 1

                start_char = word_start + local_start
                end_char = word_start + local_end

                byte_cursor = byte_end

            ids.append(token_id)
            offsets.append((start_char, end_char))

    return ids, offsets

In [ ]:
def check_offsets(tokenizer, text):
    ids, offsets = encode_with_offsets(tokenizer, text)
    assert len(ids) == len(offsets)

    for s, e in offsets:
        assert 0 <= s <= e <= len(text)

    return ids, offsets

OFFSET_TEST_STRINGS = [
    "The Supreme Court of India heard the appeal.",
    "A.B.C. v. State of X",
    "Section 302 of the Indian Penal Code, 1860.",
    "Justice R. Banumathi",
    "न्यायालय",
    "word   with\tmixed\nwhitespace",
]

for s in OFFSET_TEST_STRINGS:
    ids, offs = check_offsets(tokenizer, s)
    print(repr(s), len(ids), offs[:20])


'The Supreme Court of India heard the appeal.' 8 [(0, 3), (4, 11), (12, 17), (18, 20), (21, 26), (27, 32), (33, 36), (37, 44)]
'A.B.C. v. State of X' 7 [(0, 2), (2, 4), (4, 6), (7, 9), (10, 15), (16, 18), (19, 20)]
'Section 302 of the Indian Penal Code, 1860.' 9 [(0, 7), (8, 11), (12, 14), (15, 18), (19, 25), (26, 31), (32, 37), (38, 40), (40, 43)]
'Justice R. Banumathi' 6 [(0, 7), (8, 10), (11, 14), (14, 15), (15, 17), (17, 20)]
'न्यायालय' 25 [(0, 1), (0, 1), (0, 1), (1, 2), (1, 2), (1, 2), (2, 3), (2, 3), (2, 3), (3, 4), (3, 4), (3, 4), (4, 5), (4, 5), (4, 5), (5, 6), (5, 6), (5, 6), (6, 7), (6, 7)]
'word   with\tmixed\nwhitespace' 9 [(0, 4), (7, 11), (12, 13), (13, 17), (18, 20), (20, 22), (22, 24), (24, 25), (25, 28)]


In [ ]:
def check_gold_token_alignment(text_map, gold_map, tokenizer):
    total = 0
    exact = 0
    bad = []

    for doc_id, anns in gold_map.items():
        text = text_map[doc_id]

        ids, offsets = encode_with_offsets(tokenizer, text)

        starts = {s for s, e in offsets if s < e}
        ends   = {e for s, e in offsets if s < e}

        for ann in anns:
            start, end, label = ann[:3]
            total += 1

            start_ok = start in starts
            end_ok = end in ends

            if start_ok and end_ok:
                exact += 1
            else:
                bad.append({
                    "doc_id": doc_id,
                    "start": start,
                    "end": end,
                    "label": label,
                    "text": text[start:end],
                    "start_ok": start_ok,
                    "end_ok": end_ok,
                })

    print("Total gold entities :", total)
    print("Exactly alignable    :", exact)
    print("Not exactly alignable:", len(bad))

    if total:
        print(
            "Alignment rate      :",
            f"{100 * exact / total:.2f}%"
        )

    return bad

In [ ]:
bad1 = check_gold_token_alignment(
    fold1_text,
    fold1_gold,
    tokenizer
)

Total gold entities : 4120
Exactly alignable    : 2812
Not exactly alignable: 1308
Alignment rate      : 68.25%


In [ ]:
def inspect_bad_alignments(bad, text_map, tokenizer, n=20):
    for item in bad[:n]:
        doc_id = item["doc_id"]
        start = item["start"]
        end = item["end"]
        label = item["label"]

        text = text_map[doc_id]

        ids, offsets = encode_with_offsets(tokenizer, text)

        print("=" * 80)
        print("DOC:", doc_id)
        print("LABEL:", label)
        print("GOLD:", repr(text[start:end]))
        print("SPAN:", (start, end))
        print(
            "start_ok:", item["start_ok"],
            "| end_ok:", item["end_ok"]
        )

        lo = max(0, start - 30)
        hi = min(len(text), end + 30)

        print("CONTEXT:")
        print(repr(text[lo:hi]))

        print("\nNearby tokens:")

        for token_id, (s, e) in zip(ids, offsets):
            if e >= start - 5 and s <= end + 5:
                print(
                    repr(text[s:e]),
                    (s, e),
                    tokenizer.itos[token_id]
                )

        print()

In [ ]:
inspect_bad_alignments(
    bad1,
    fold1_text,
    tokenizer,
    n=20
)

DOC: 115651329
LABEL: DATE
GOLD: '18.11.1994'
SPAN: (575, 585)
start_ok: True | end_ok: False
CONTEXT:
' that on the fateful day i.e. 18.11.1994, at about 8.00 A.M. in the mo'

Nearby tokens:
'i.e.' (570, 574) <0x692e652e</w>>
'18.' (575, 578) <0x31382e>
'11.' (578, 581) <0x31312e>
'199' (581, 584) <0x313939>
'4,' (584, 586) <0x342c</w>>
'at' (587, 589) <0x6174</w>>
'about' (590, 595) <0x61626f7574</w>>

DOC: 115651329
LABEL: WIT
GOLD: 'Pardeep Kumar'
SPAN: (1809, 1822)
start_ok: True | end_ok: False
CONTEXT:
'ealth Centre, Taoru where Dr. Pardeep Kumar, Medical Officer, medically e'

Nearby tokens:
'where' (1799, 1804) <0x7768657265</w>>
'Dr.' (1805, 1808) <0x44722e</w>>
'Par' (1809, 1812) <0x506172>
'deep' (1812, 1816) <0x64656570</w>>
'Kumar,' (1817, 1823) <0x4b756d61722c</w>>
'Medical' (1824, 1831) <0x4d65646963616c</w>>

DOC: 115651329
LABEL: STAT
GOLD: 'IPC'
SPAN: (2491, 2494)
start_ok: True | end_ok: False
CONTEXT:
' and 307 read with Section 34 IPC. PW 8-Ramesh Kumar (ASI) had 

## 4. NER target formulation

The course allows token classification, span classification, or span extraction.

If you choose BIO token classification, one possible vocabulary shape is:

`O, B-COURT, I-COURT, B-JUDGE, I-JUDGE, ...`

The final output must still be exact character spans. This notebook uses BIO token classification plus start/end boundary-correction heads, decoded back to exact character spans.


In [ ]:
entity_labels = [
    "APP",
    "RESP",
    "A.COUNSEL",
    "R.COUNSEL",
    "JUDGE",
    "WIT",
    "AUTH",
    "COURT",
    "STAT",
    "PREC",
    "DATE",
    "CASENO",
]

bio_labels = ["O"]

for label in entity_labels:
    bio_labels.append(f"B-{label}")
    bio_labels.append(f"I-{label}")

label2id = {
    label: idx
    for idx, label in enumerate(bio_labels)
}

id2label = {
    idx: label
    for label, idx in label2id.items()
}

print("Number of BIO classes:", len(bio_labels))
print(label2id)
print(id2label)

Number of BIO classes: 25
{'O': 0, 'B-APP': 1, 'I-APP': 2, 'B-RESP': 3, 'I-RESP': 4, 'B-A.COUNSEL': 5, 'I-A.COUNSEL': 6, 'B-R.COUNSEL': 7, 'I-R.COUNSEL': 8, 'B-JUDGE': 9, 'I-JUDGE': 10, 'B-WIT': 11, 'I-WIT': 12, 'B-AUTH': 13, 'I-AUTH': 14, 'B-COURT': 15, 'I-COURT': 16, 'B-STAT': 17, 'I-STAT': 18, 'B-PREC': 19, 'I-PREC': 20, 'B-DATE': 21, 'I-DATE': 22, 'B-CASENO': 23, 'I-CASENO': 24}
{0: 'O', 1: 'B-APP', 2: 'I-APP', 3: 'B-RESP', 4: 'I-RESP', 5: 'B-A.COUNSEL', 6: 'I-A.COUNSEL', 7: 'B-R.COUNSEL', 8: 'I-R.COUNSEL', 9: 'B-JUDGE', 10: 'I-JUDGE', 11: 'B-WIT', 12: 'I-WIT', 13: 'B-AUTH', 14: 'I-AUTH', 15: 'B-COURT', 16: 'I-COURT', 17: 'B-STAT', 18: 'I-STAT', 19: 'B-PREC', 20: 'I-PREC', 21: 'B-DATE', 22: 'I-DATE', 23: 'B-CASENO', 24: 'I-CASENO'}


In [ ]:
START_DELTAS = [0, 1, 3]
END_DELTAS = [-1, 0, 1, 2, 3, 4]

start_delta2id = {
    d: i for i, d in enumerate(START_DELTAS)
}

start_id2delta = {
    i: d for d, i in start_delta2id.items()
}

end_delta2id = {
    d: i for i, d in enumerate(END_DELTAS)
}

end_id2delta = {
    i: d for d, i in end_delta2id.items()
}

In [ ]:
def build_token_targets(text, token_offsets, gold_spans):
    bio_targets = ["O"] * len(token_offsets)

    start_targets = [-100] * len(token_offsets)
    end_targets = [-100] * len(token_offsets)

    for start, end, label in gold_spans:

        entity_token_ids = []

        for i, (ts, te) in enumerate(token_offsets):
            if ts == te:
                continue

            if te > start and ts < end:
                entity_token_ids.append(i)

        if not entity_token_ids:
            continue

        first = entity_token_ids[0]
        last = entity_token_ids[-1]

        # -------------------------
        # BIO targets
        # -------------------------
        bio_targets[first] = f"B-{label}"

        for i in entity_token_ids[1:]:
            bio_targets[i] = f"I-{label}"

        # -------------------------
        # Boundary targets
        # -------------------------
        first_ts, first_te = token_offsets[first]
        last_ts, last_te = token_offsets[last]

        start_delta = start - first_ts
        end_delta = last_te - end

        if start_delta not in start_delta2id:
            raise ValueError(
                f"Unknown start delta {start_delta} "
                f"for entity {text[start:end]!r}"
            )

        if end_delta not in end_delta2id:
            raise ValueError(
                f"Unknown end delta {end_delta} "
                f"for entity {text[start:end]!r}"
            )

        start_targets[first] = start_delta2id[start_delta]
        end_targets[last] = end_delta2id[end_delta]

    bio_targets = [
        label2id[x]
        for x in bio_targets
    ]

    return bio_targets, start_targets, end_targets

In [ ]:
doc_id = "115651329"

text = fold1_text[doc_id]

ids, offsets = encode_with_offsets(
    tokenizer,
    text
)

gold_spans = normalize_gold(
    fold1_gold[doc_id]
)

bio_targets, start_targets, end_targets = build_token_targets(
    text,
    offsets,
    gold_spans
)

print("Tokens :", len(ids))
print("BIO    :", len(bio_targets))
print("Start  :", len(start_targets))
print("End    :", len(end_targets))

print(
    "Start boundary labels:",
    sum(x != -100 for x in start_targets)
)

print(
    "End boundary labels:",
    sum(x != -100 for x in end_targets)
)

Tokens : 4051
BIO    : 4051
Start  : 4051
End    : 4051
Start boundary labels: 104
End boundary labels: 104


## 5. Long-document windowing

Keep original character offsets attached to every token/window.

Suggested record:

```python
{
    "doc_id": ...,
    "input_ids": ...,
    "attention_mask": ...,
    "token_offsets": ...,
    "targets": ...
}
```

If you use overlapping windows, define how duplicate predictions will be merged.


In [ ]:
def make_windows(
    doc_id,
    text,
    gold_spans,
    tokenizer,
    max_tokens=2048,
    stride=1536,
):
    input_ids, token_offsets = encode_with_offsets(
        tokenizer,
        text
    )

    (
        bio_targets,
        start_targets,
        end_targets,
    ) = build_token_targets(
        text,
        token_offsets,
        gold_spans
    )

    windows = []

    n = len(input_ids)
    window_id = 0

    for st in range(0, n, stride):
        en = min(st + max_tokens, n)

        ids = input_ids[st:en]
        offs = token_offsets[st:en]

        bio = bio_targets[st:en]
        start_b = start_targets[st:en]
        end_b = end_targets[st:en]

        # ---------------------------------
        # Left boundary:
        # window begins in middle of entity
        # ---------------------------------
        if st > 0:
            first_tag = id2label[bio[0]]

            if first_tag.startswith("I-"):
                label = first_tag[2:]

                i = 0

                while i < len(bio):
                    tag = id2label[bio[i]]

                    if tag != f"I-{label}":
                        break

                    bio[i] = -100
                    start_b[i] = -100
                    end_b[i] = -100

                    i += 1

        # ---------------------------------
        # Right boundary:
        # window ends in middle of entity
        # ---------------------------------
        if en < n:
            next_tag = id2label[bio_targets[en]]

            if next_tag.startswith("I-"):
                label = next_tag[2:]

                i = len(bio) - 1

                while i >= 0:
                    tag = id2label[bio[i]]

                    if tag not in {
                        f"B-{label}",
                        f"I-{label}",
                    }:
                        break

                    bio[i] = -100
                    start_b[i] = -100
                    end_b[i] = -100

                    i -= 1

        char_start = (
            offs[0][0]
            if len(offs) > 0
            else 0
        )

        char_end = (
            offs[-1][1]
            if len(offs) > 0
            else 0
        )

        windows.append({
            "doc_id": doc_id,
            "window_id": window_id,

            "token_start": st,
            "token_end": en,

            "char_start": char_start,
            "char_end": char_end,

            "input_ids": ids,
            "attention_mask": [1] * len(ids),

            "token_offsets": offs,

            "targets": bio,
            "start_targets": start_b,
            "end_targets": end_b,
        })

        window_id += 1

        if en == n:
            break

    return windows

In [ ]:
#create windows

def build_all_windows(
    text_map,
    gold_map,
    tokenizer,
    max_tokens=2048,
    stride=1536,
):
    all_windows = []

    for doc_id, text in text_map.items():

        spans = normalize_gold(
            gold_map.get(doc_id, [])
        )

        doc_windows = make_windows(
            doc_id=doc_id,
            text=text,
            tokenizer=tokenizer,
            gold_spans=spans,
            max_tokens=max_tokens,
            stride=stride,
        )

        all_windows.extend(doc_windows)

    return all_windows

In [ ]:
doc_id = "115651329"

wins = make_windows(
    doc_id=doc_id,
    text=fold1_text[doc_id],
    gold_spans=normalize_gold(
        fold1_gold[doc_id]
    ),
    tokenizer=tokenizer,
    max_tokens=2048,
    stride=1536,
)

for w in wins:
    print(
        "window", w["window_id"],
        "| tokens", w["token_start"], w["token_end"],
        "| BIO supervised:",
        sum(x != -100 for x in w["targets"]),
        "| start boundaries:",
        sum(x != -100 for x in w["start_targets"]),
        "| end boundaries:",
        sum(x != -100 for x in w["end_targets"]),
    )

window 0 | tokens 0 2048 | BIO supervised: 2048 | start boundaries: 60 | end boundaries: 60
window 1 | tokens 1536 3584 | BIO supervised: 2048 | start boundaries: 35 | end boundaries: 35
window 2 | tokens 3072 4051 | BIO supervised: 979 | start boundaries: 25 | end boundaries: 25


In [ ]:
fold1_windows = build_all_windows(
    fold1_text,
    fold1_gold,
    tokenizer,
    max_tokens=2048,
    stride=1536,
)

fold2_windows = build_all_windows(
    fold2_text,
    fold2_gold,
    tokenizer,
    max_tokens=2048,
    stride=1536,
)
train_windows = fold1_windows + fold2_windows
print("Fold 1 windows:", len(fold1_windows))
print("Fold 2 windows:", len(fold2_windows))
print("Total windows :", len(train_windows))
w = train_windows[0]

print("doc_id:", w["doc_id"])
print("window_id:", w["window_id"])
print("num tokens:", len(w["input_ids"]))
print("num targets:", len(w["targets"]))
print("char range:", (w["char_start"], w["char_end"]))
print(
    "ignored:",
    sum(x == -100 for x in w["targets"])
)
assert all(
    len(w["input_ids"]) == len(w["targets"])
    for w in train_windows
)

assert all(
    len(w["input_ids"]) <= 2048
    for w in train_windows
)

print("All window sanity checks passed.")

Fold 1 windows: 155
Fold 2 windows: 157
Total windows : 312
doc_id: 115651329
window_id: 0
num tokens: 2048
num targets: 2048
char range: (0, 9406)
ignored: 0
All window sanity checks passed.


In [ ]:
import random

random.seed(42)

all_doc_ids = list(
    set(fold1_text.keys()) |
    set(fold2_text.keys())
)

random.shuffle(all_doc_ids)

val_fraction = 0.15

n_val = int(len(all_doc_ids) * val_fraction)

val_doc_ids = set(all_doc_ids[:n_val])
train_doc_ids = set(all_doc_ids[n_val:])

train_windows_split = [
    w for w in train_windows
    if w["doc_id"] in train_doc_ids
]

val_windows_split = [
    w for w in train_windows
    if w["doc_id"] in val_doc_ids
]

print("Train docs   :", len(train_doc_ids))
print("Validation docs:", len(val_doc_ids))

print("Train windows:", len(train_windows_split))
print("Val windows  :", len(val_windows_split))

Train docs   : 60
Validation docs: 10
Train windows: 267
Val windows  : 45


In [ ]:
train_ids_check = {
    w["doc_id"]
    for w in train_windows_split
}

val_ids_check = {
    w["doc_id"]
    for w in val_windows_split
}

assert train_ids_check.isdisjoint(val_ids_check)

print("No document leakage.")

No document leakage.


In [ ]:
from collections import Counter
import torch


def make_class_weights(
    windows,
    key,
    num_classes,
    ignore_index=-100,
    power=0.5,
    max_weight=8.0,
):
    cnt = Counter()

    for w in windows:
        for x in w[key]:
            if x != ignore_index:
                cnt[int(x)] += 1

    counts = torch.tensor(
        [cnt.get(i, 0) for i in range(num_classes)],
        dtype=torch.float
    )

    weights = torch.zeros_like(counts)

    nz = counts > 0
    weights[nz] = 1.0 / counts[nz].pow(power)

    # normalize around 1
    weights[nz] /= weights[nz].mean()

    weights = torch.clamp(
        weights,
        max=max_weight
    )

    return weights, counts

In [ ]:
bio_weights, bio_counts = make_class_weights(
    train_windows_split,
    "targets",
    len(label2id),
    power=0.5,
    max_weight=8.0,
)

start_weights, start_counts = make_class_weights(
    train_windows_split,
    "start_targets",
    len(START_DELTAS),
    power=0.5,
    max_weight=8.0,
)

end_weights, end_counts = make_class_weights(
    train_windows_split,
    "end_targets",
    len(END_DELTAS),
    power=0.5,
    max_weight=8.0,
)

print("BIO counts:", bio_counts.tolist())
print("BIO weights:", bio_weights.tolist())

print("START counts:", start_counts.tolist())
print("START weights:", start_weights.tolist())

print("END counts:", end_counts.tolist())
print("END weights:", end_weights.tolist())

BIO counts: [450811.0, 404.0, 1603.0, 373.0, 1359.0, 213.0, 662.0, 187.0, 617.0, 223.0, 1128.0, 223.0, 666.0, 1111.0, 1704.0, 1089.0, 1718.0, 1385.0, 3682.0, 1281.0, 12750.0, 1709.0, 4207.0, 694.0, 4467.0]
BIO weights: [0.043311893939971924, 1.4468176364898682, 0.7263361811637878, 1.5057401657104492, 0.7888505458831787, 1.9925748109817505, 1.1302523612976074, 2.126589298248291, 1.170743703842163, 1.9473859071731567, 0.8658643960952759, 1.9473859071731567, 1.126853108406067, 0.8724638223648071, 0.7044816017150879, 0.881232500076294, 0.7016053199768066, 0.7814110517501831, 0.4792504608631134, 0.8125120997428894, 0.25754275918006897, 0.7034502625465393, 0.44835105538368225, 1.1038873195648193, 0.43510740995407104]
START counts: [8852.0, 38.0, 2.0]
START weights: [0.036235909909009933, 0.553054690361023, 2.4107093811035156]
END counts: [19.0, 6205.0, 2623.0, 42.0, 1.0, 2.0]
END weights: [0.648358166217804, 0.035877395421266556, 0.055181391537189484, 0.43608102202415466, 2.826127767562866, 

In [ ]:
class NERWindowDataset(Dataset):
    def __init__(self, windows):
        self.windows = windows

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        w = self.windows[idx]

        return {
            "input_ids": w["input_ids"],
            "attention_mask": w["attention_mask"],

            "labels": w["targets"],
            "start_labels": w["start_targets"],
            "end_labels": w["end_targets"],

            "doc_id": w["doc_id"],
            "window_id": w["window_id"],

            "token_start": w["token_start"],
            "token_end": w["token_end"],

            "token_offsets": w["token_offsets"],
        }

In [ ]:
train_dataset_ner = NERWindowDataset(
    train_windows_split
)

val_dataset_ner = NERWindowDataset(
    val_windows_split
)

print("Train samples:", len(train_dataset_ner))
print("Val samples  :", len(val_dataset_ner))

Train samples: 267
Val samples  : 45


In [ ]:
def ner_collator(batch):
    max_len = max(
        len(x["input_ids"])
        for x in batch
    )

    input_ids = []
    attention_mask = []

    labels = []
    start_labels = []
    end_labels = []

    doc_ids = []
    window_ids = []

    token_starts = []
    token_ends = []

    token_offsets = []

    for x in batch:
        n = len(x["input_ids"])
        pad_len = max_len - n

        input_ids.append(
            x["input_ids"] +
            [PAD_ID] * pad_len
        )

        attention_mask.append(
            x["attention_mask"] +
            [0] * pad_len
        )

        labels.append(
            x["labels"] +
            [-100] * pad_len
        )

        start_labels.append(
            x["start_labels"] +
            [-100] * pad_len
        )

        end_labels.append(
            x["end_labels"] +
            [-100] * pad_len
        )

        doc_ids.append(x["doc_id"])
        window_ids.append(x["window_id"])

        token_starts.append(x["token_start"])
        token_ends.append(x["token_end"])

        token_offsets.append(
            x["token_offsets"]
        )

    return {
        "input_ids": torch.tensor(
            input_ids,
            dtype=torch.long
        ),

        "attention_mask": torch.tensor(
            attention_mask,
            dtype=torch.long
        ),

        "labels": torch.tensor(
            labels,
            dtype=torch.long
        ),

        "start_labels": torch.tensor(
            start_labels,
            dtype=torch.long
        ),

        "end_labels": torch.tensor(
            end_labels,
            dtype=torch.long
        ),

        "doc_id": doc_ids,
        "window_id": window_ids,

        "token_start": token_starts,
        "token_end": token_ends,

        "token_offsets": token_offsets,
    }

## 6A. Decoder-only Transformer (Phase-1 backbone)

Same configuration as the Phase-1 checkpoint. `model.encode(...)` returns the final hidden states `[B, L, D]` (before the LM head), which feed the NER heads.


In [ ]:
import math
from dataclasses import asdict, dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class DecoderConfig:
    vocab_size: int
    context_length: int
    n_embd: int
    n_layer: int
    n_head: int
    embd_pdrop: float = 0.1
    resid_pdrop: float = 0.1
    attn_pdrop: float = 0.1
    layer_norm_epsilon: float = 1e-5
    initializer_range: float = 0.02
    bos_token_id: int = 2
    eos_token_id: int = 3
    pad_token_id: int = 0
    n_inner: int | None = None
    activation_function: str = "gelu_new"
    scale_attn_weights: bool = True
    scale_attn_by_inverse_layer_idx: bool = False
    tie_word_embeddings: bool = True

    @property
    def feed_forward_size(self):
        return self.n_inner if self.n_inner is not None else 4 * self.n_embd


class CausalSelfAttention(nn.Module):
    """Causal multi-head self-attention with a fused QKV projection."""

    def __init__(self, config: DecoderConfig, layer_idx: int):
        super().__init__()
        assert config.n_embd % config.n_head == 0, (
            "n_embd must be divisible by n_head"
        )
        self.n_head = config.n_head
        self.head_dim = config.n_embd // config.n_head
        self.layer_idx = layer_idx
        self.scale_attn_weights = config.scale_attn_weights
        self.scale_attn_by_inverse_layer_idx = (
            config.scale_attn_by_inverse_layer_idx
        )

        self.qkv = nn.Linear(config.n_embd, 3 * config.n_embd, bias=True)
        self.out_proj = nn.Linear(config.n_embd, config.n_embd, bias=True)
        self.attn_dropout = nn.Dropout(config.attn_pdrop)
        self.resid_dropout = nn.Dropout(config.resid_pdrop)

    def forward(self, hidden_states, attention_mask=None):
            batch_size, seq_len, hidden_size = hidden_states.shape
        
            qkv = self.qkv(hidden_states)
            query, key, value = qkv.split(hidden_size, dim=-1)
        
            def split_heads(tensor):
                return tensor.view(
                    batch_size,
                    seq_len,
                    self.n_head,
                    self.head_dim
                ).transpose(1, 2)
        
            query = split_heads(query)
            key = split_heads(key)
            value = split_heads(value)
        
            scale = None
        
            if self.scale_attn_weights:
                scale = 1.0 / math.sqrt(self.head_dim)
        
            if self.scale_attn_by_inverse_layer_idx:
                if scale is None:
                    scale = 1.0
                scale = scale / float(self.layer_idx + 1)
        
            attn_mask = None
            is_causal = True
        
            if attention_mask is not None:
                key_is_valid = attention_mask.to(
                    device=hidden_states.device,
                    dtype=torch.bool
                )
        
                # [B, 1, 1, L]
                attn_mask = key_is_valid[:, None, None, :]
        
                # Since an explicit mask is supplied, combine causal mask manually
                causal = torch.ones(
                    seq_len,
                    seq_len,
                    dtype=torch.bool,
                    device=hidden_states.device
                ).tril()
        
                attn_mask = attn_mask & causal[None, None, :, :]
                is_causal = False
        
            attended = F.scaled_dot_product_attention(
                query,
                key,
                value,
                attn_mask=attn_mask,
                dropout_p=self.attn_dropout.p if self.training else 0.0,
                is_causal=is_causal,
                scale=scale,
            )
        
            attended = attended.transpose(1, 2).contiguous().view(
                batch_size,
                seq_len,
                hidden_size
            )
        
            return self.resid_dropout(
                self.out_proj(attended)
            )


class DecoderMLP(nn.Module):
    """Position-wise feed-forward block (expand -> activation -> project)."""

    def __init__(self, config: DecoderConfig):
        super().__init__()
        self.fc_in = nn.Linear(
            config.n_embd, config.feed_forward_size, bias=True
        )
        self.fc_out = nn.Linear(
            config.feed_forward_size, config.n_embd, bias=True
        )
        self.dropout = nn.Dropout(config.resid_pdrop)
        self.activation_name = config.activation_function

    def _activation(self, x):
        if self.activation_name == "gelu_new":
            # Tanh approximation of GELU.
            return 0.5 * x * (
                1.0
                + torch.tanh(
                    math.sqrt(2.0 / math.pi)
                    * (x + 0.044715 * torch.pow(x, 3.0))
                )
            )
        if self.activation_name == "gelu":
            return F.gelu(x)
        if self.activation_name == "relu":
            return F.relu(x)
        if self.activation_name in ("silu", "swish"):
            return F.silu(x)
        raise ValueError(
            f"Unsupported activation: {self.activation_name}"
        )

    def forward(self, hidden_states):
        return self.dropout(self.fc_out(self._activation(self.fc_in(hidden_states))))


class DecoderBlock(nn.Module):
    """Pre-norm Transformer block with residual attention and MLP paths."""

    def __init__(self, config: DecoderConfig, layer_idx: int):
        super().__init__()
        self.ln_1 = nn.LayerNorm(
            config.n_embd, eps=config.layer_norm_epsilon
        )
        self.attn = CausalSelfAttention(config, layer_idx)
        self.ln_2 = nn.LayerNorm(
            config.n_embd, eps=config.layer_norm_epsilon
        )
        self.mlp = DecoderMLP(config)

    def forward(self, hidden_states, attention_mask=None):
        hidden_states = hidden_states + self.attn(
            self.ln_1(hidden_states), attention_mask=attention_mask
        )
        hidden_states = hidden_states + self.mlp(self.ln_2(hidden_states))
        return hidden_states


class DecoderOnlyTransformer(nn.Module):
    """Decoder-only causal language model implemented in raw PyTorch."""

    def __init__(self, config: DecoderConfig):
        super().__init__()
        if config.n_embd % config.n_head != 0:
            raise ValueError("n_embd must be divisible by n_head")
        if config.context_length <= 0:
            raise ValueError("context_length must be positive")
        self.config = config
        self.token_embedding = nn.Embedding(config.vocab_size, config.n_embd)
        self.position_embedding = nn.Embedding(
            config.context_length, config.n_embd
        )
        self.embedding_dropout = nn.Dropout(config.embd_pdrop)
        self.blocks = nn.ModuleList(
            DecoderBlock(config, layer_idx)
            for layer_idx in range(config.n_layer)
        )
        self.final_ln = nn.LayerNorm(
            config.n_embd, eps=config.layer_norm_epsilon
        )
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        self.apply(self._initialize_weights)
        if config.tie_word_embeddings:
            # Weight tying: the output projection shares the token-embedding matrix.
            self.lm_head.weight = self.token_embedding.weight

    def _initialize_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=self.config.initializer_range)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.LayerNorm):
            nn.init.ones_(module.weight)
            nn.init.zeros_(module.bias)

    def _validate_inputs(self, input_ids, attention_mask=None):
        if input_ids.ndim != 2:
            raise ValueError("input_ids must have shape [batch, sequence]")
        batch_size, seq_len = input_ids.shape
        if seq_len == 0:
            raise ValueError("input sequence must not be empty")
        assert seq_len <= self.config.context_length, (
            f"sequence length {seq_len} exceeds context length "
            f"{self.config.context_length}"
        )
        if attention_mask is not None:
            if attention_mask.shape != (batch_size, seq_len):
                raise ValueError(
                    "attention_mask must have shape [batch, sequence]"
                )
        return batch_size, seq_len

    def _embed(self, input_ids):
        _, seq_len = input_ids.shape
        position_ids = torch.arange(
            seq_len, dtype=torch.long, device=input_ids.device
        ).unsqueeze(0)
        return self.embedding_dropout(
            self.token_embedding(input_ids)
            + self.position_embedding(position_ids)
        )

    def encode(self, input_ids, attention_mask=None):
        """Return final normalized hidden states with shape [B, L, D]."""
        self._validate_inputs(input_ids, attention_mask)
        hidden_states = self._embed(input_ids)
        for block in self.blocks:
            hidden_states = block(hidden_states, attention_mask)
        return self.final_ln(hidden_states)

    def debug_hidden_states(self, input_ids, attention_mask=None):
        """Expose intermediate states for optional migration debugging."""
        self._validate_inputs(input_ids, attention_mask)
        hidden_states = self._embed(input_ids)
        states = {"embeddings": hidden_states}
        for layer_idx, block in enumerate(self.blocks):
            hidden_states = block(hidden_states, attention_mask)
            states[f"block_{layer_idx}"] = hidden_states
        states["final_norm"] = self.final_ln(hidden_states)
        return states

    def forward(self, input_ids, attention_mask=None, labels=None):
        hidden_states = self.encode(input_ids, attention_mask)
        logits = self.lm_head(hidden_states)
        loss = None

        if labels is not None:
            if labels.shape != input_ids.shape:
                raise ValueError("labels must have the same shape as input_ids")
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()
            valid_labels = shift_labels.ne(-100)
            if attention_mask is not None:
                valid_labels = valid_labels & attention_mask[:, 1:].bool()
            
            shift_labels = shift_labels.masked_fill(~valid_labels, -100)
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1),
                ignore_index=-100,
            )

        return {
            "loss": loss,
            "logits": logits,
            "hidden_states": hidden_states,
        }





In [ ]:
decoder_config = DecoderConfig(
    vocab_size=16000,
    context_length=2048,
    n_embd=640,
    n_layer=8,
    n_head=8,
    embd_pdrop=0.1,
    resid_pdrop=0.1,
    attn_pdrop=0.1,
    layer_norm_epsilon=1e-5,
    initializer_range=0.02,
    bos_token_id=BOS_ID,
    eos_token_id=EOS_ID,
    pad_token_id=PAD_ID,
    n_inner=None,
    activation_function="gelu_new",
    scale_attn_weights=True,
    scale_attn_by_inverse_layer_idx=False,
    tie_word_embeddings=True,
)

In [ ]:
# Sanity check: the Phase-1 decoder checkpoint loads strictly into this class.
# Done on CPU and released; load_backbone() builds the copy that is trained.
DECODER_CHECKPOINT = "/kaggle/input/datasets/sg123321/deconly-model/pytorch_model.bin"

decoder_backbone = DecoderOnlyTransformer(decoder_config)
state = torch.load(DECODER_CHECKPOINT, map_location="cpu")
missing, unexpected = decoder_backbone.load_state_dict(state, strict=True)

print("missing:", missing)
print("unexpected:", unexpected)

del decoder_backbone, state


## 6B. Diagonal SSM / S4D-inspired LM (Phase-1 backbone)

Same `DiagonalSSM`, `SSMBlock` and `SSMLM` classes as Phase 1. `encode()` returns the representation after `norm_f` and before `lm_head`.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
class SSMLM(nn.Module):
    def __init__(
        self,
        vocab_size,
        dim,
        n_layer,
        max_seq_len,
        state_dim=16,
        expand=3,
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.dim = dim
        self.n_layer = n_layer
        self.max_seq_len = max_seq_len

        self.embedding = nn.Embedding(
            vocab_size,
            dim
        )

        self.layers = nn.ModuleList([
            SSMBlock(
                dim=dim,
                state_dim=state_dim,
                expand=expand,
            )
            for _ in range(n_layer)
        ])

        self.norm_f = nn.LayerNorm(dim)

        self.lm_head = nn.Linear(
            dim,
            vocab_size,
            bias=False
        )

    def forward(
        self,
        input_ids,
        labels=None,
        **kwargs
    ):
        B, T = input_ids.shape

        if T > self.max_seq_len:
            raise ValueError(
                f"Sequence length {T} exceeds "
                f"max_seq_len={self.max_seq_len}"
            )

        x = self.embedding(input_ids)

        for layer in self.layers:
            x = layer(x)

        x = self.norm_f(x)

        logits = self.lm_head(x)

        if labels is not None:

            targets = torch.full_like(
                labels,
                -100
            )

            targets[:, :-1] = labels[:, 1:]

            loss = F.cross_entropy(
                logits.reshape(
                    -1,
                    self.vocab_size
                ),
                targets.reshape(-1),
                ignore_index=-100,
            )

            return {
                "loss": loss
            }

        return {
            "logits": logits
        }
class SSMBlock(nn.Module):
    def __init__(
        self,
        dim,
        state_dim,
        expand=3,
    ):
        super().__init__()

        self.norm = nn.LayerNorm(dim)

        self.ssm = DiagonalSSM(
            dim=dim,
            state_dim=state_dim,
            expand=expand,
        )

    def forward(self, x):
        return x + self.ssm(self.norm(x))
class DiagonalSSM(nn.Module):
    """
    S4D-inspired diagonal state-space layer.

    Input : [B, T, C]
    Output: [B, T, C]
    """

    def __init__(
        self,
        dim,
        state_dim,
        expand=3,
    ):
        super().__init__()

        self.dim = dim
        self.state_dim = state_dim
        self.inner_dim = dim * expand

        # content + gate
        self.in_proj = nn.Linear(
            dim,
            2 * self.inner_dim
        )

        # Stable diagonal state transition
        # A will become negative
        A = torch.arange(
            1,
            state_dim + 1,
            dtype=torch.float32
        )

        self.A_log = nn.Parameter(
            torch.log(A)
        )

        # Per-channel SSM input/output weights
        self.B = nn.Parameter(
            torch.randn(
                self.inner_dim,
                state_dim
            ) * 0.02
        )

        self.C = nn.Parameter(
            torch.randn(
                self.inner_dim,
                state_dim
            ) * 0.02
        )

        # Per-channel time scale
        self.log_dt = nn.Parameter(
            torch.zeros(self.inner_dim)
        )

        # Direct skip term
        self.D = nn.Parameter(
            torch.ones(self.inner_dim)
        )

        self.out_proj = nn.Linear(
            self.inner_dim,
            dim
        )

    def forward(self, x):
        Bsz, T, _ = x.shape

        xz = self.in_proj(x)

        u, z = xz.chunk(
            2,
            dim=-1
        )

        # -------------------------------------------------
        # Step 1
        #
        # Construct stable continuous transition:
        #
        # A = -exp(A_log)
        #
        # shape:
        # [state_dim]
        # -------------------------------------------------

        A = -torch.exp(self.A_log)

        # -------------------------------------------------
        # Step 2
        #
        # dt must always be positive.
        #
        # use softplus(log_dt)
        #
        # shape:
        # [inner_dim]
        # -------------------------------------------------

        dt = F.softplus(self.log_dt)

        # -------------------------------------------------
        # Step 3
        #
        # Create discrete transition:
        #
        # A_bar[c,n] = exp(dt[c] * A[n])
        #
        # expected:
        # [inner_dim, state_dim]
        # -------------------------------------------------

        A_bar = torch.exp(
            dt.unsqueeze(-1) * A.unsqueeze(0)
        )

        # -------------------------------------------------
        # Step 4
        #
        # Zero-order-hold style input coefficient:
        #
        # B_bar[c,n]
        #   = ((A_bar[c,n] - 1) / A[n]) * B[c,n]
        #
        # expected:
        # [inner_dim, state_dim]
        # -------------------------------------------------

        xA = dt.unsqueeze(-1) * A.unsqueeze(0)

        B_bar = (
            torch.expm1(xA) / A.unsqueeze(0)
        ) * self.B

        # -------------------------------------------------
        # Step 5
        #
        # Build time indices:
        #
        # t = [0, 1, ..., T-1]
        #
        # Then construct the impulse-response kernel:
        #
        # K[c,t] =
        #   sum_n C[c,n]
        #         * A_bar[c,n]^t
        #         * B_bar[c,n]
        #
        # Target shape:
        # [inner_dim, T]
        #
        # Hint:
        # avoid a Python loop over T.
        # Broadcasting / einsum should be used.
        # -------------------------------------------------

        t = torch.arange(
            T,
            device=x.device,
            dtype=A_bar.dtype
        )

        powers = A_bar.unsqueeze(-1) ** t
        # [inner_dim, state_dim, T]

        kernel = torch.einsum(
            "cn,cnt->ct",
            self.C * B_bar,
            powers
        )
        # [inner_dim, T]

        # -------------------------------------------------
        # Step 6 — FFT causal convolution
        #
        # u currently:
        # [B, T, inner_dim]
        #
        # transpose to:
        # [B, inner_dim, T]
        #
        # Use an FFT length >= 2*T.
        #
        # rfft(input)
        # rfft(kernel)
        # multiply
        # irfft
        #
        # finally truncate to first T values.
        #
        # This gives causal convolution:
        #
        # y[t] = sum_{i=0}^{t} K[i] u[t-i]
        # -------------------------------------------------

        u = u.transpose(1, 2)
        # [B, inner_dim, T]

        fft_len = 2 * T

        u_fft = u.float()
        kernel_fft = kernel.float()
        
        u_f = torch.fft.rfft(
            u_fft,
            n=fft_len
        )
        
        k_f = torch.fft.rfft(
            kernel_fft,
            n=fft_len
        )
        
        y = torch.fft.irfft(
            u_f * k_f.unsqueeze(0),
            n=fft_len
        )[..., :T]
        
        y = y.to(u.dtype)

        # -------------------------------------------------
        # Step 7
        #
        # Add direct SSM skip:
        #
        # y += D * u
        # -------------------------------------------------

        y = y + self.D.view(1, -1, 1) * u

        # restore:
        # [B, T, inner_dim]

        y = y.transpose(1, 2)

        # same gating idea as before
        y = y * F.silu(z)

        return self.out_proj(y)

In [ ]:
def ssm_encode(self, input_ids, attention_mask=None):
    B, T = input_ids.shape

    if T > self.max_seq_len:
        raise ValueError(
            f"Sequence length {T} exceeds max_seq_len={self.max_seq_len}"
        )

    x = self.embedding(input_ids)

    for layer in self.layers:
        x = layer(x)

    x = self.norm_f(x)

    return x

SSMLM.encode = ssm_encode

In [ ]:
SSM_DIM = 700
SSM_N_LAYER = 10
SSM_MAX_SEQ_LEN = 2048

SSM_STATE_DIM = 16
SSM_EXPAND = 3
SSM_CHECKPOINT = "/kaggle/input/datasets/sg123321/ssmmodel/model.safetensors"
def build_ssm_backbone():
    return SSMLM(
        vocab_size=16000,
        dim=SSM_DIM,
        n_layer=SSM_N_LAYER,
        max_seq_len=SSM_MAX_SEQ_LEN,
        state_dim=SSM_STATE_DIM,
        expand=SSM_EXPAND,
    )


## 7. Architecture registry

Registered backbones: `decoder` and `ssm`. The rest of the notebook is backbone-agnostic.


In [ ]:
# ============================================================
# 7. GENERIC BACKBONE REGISTRY
# ============================================================

BACKBONES = {}


def register_backbone(
    name,
    builder,
    hidden_dim,
    checkpoint_path=None,
):
    BACKBONES[name] = {
        "builder": builder,
        "hidden_dim": hidden_dim,
        "checkpoint_path": checkpoint_path,
    }


def load_backbone(
    name,
    checkpoint_path=None,
    device="cpu",
):
    if name not in BACKBONES:
        raise ValueError(
            f"Unknown backbone: {name}. "
            f"Available: {list(BACKBONES.keys())}"
        )

    info = BACKBONES[name]

    model = info["builder"]()

    ckpt = (
        checkpoint_path
        if checkpoint_path is not None
        else info["checkpoint_path"]
    )

    if ckpt is None:
        raise ValueError(
            f"No checkpoint supplied for backbone '{name}'"
        )

    if str(ckpt).endswith(".safetensors"):
        from safetensors.torch import load_file
        state = load_file(str(ckpt), device="cpu")
    else:
        state = torch.load(
            ckpt,
            map_location="cpu"
        )

    # Support either:
    # 1. raw state_dict
    # 2. {"model_state_dict": ...}
    if (
        isinstance(state, dict)
        and "model_state_dict" in state
    ):
        state = state["model_state_dict"]

    result = model.load_state_dict(
        state,
        strict=True
    )

    print("Backbone:", name)
    print("Missing keys:", result.missing_keys)
    print("Unexpected keys:", result.unexpected_keys)

    model = model.to(device)

    return model

In [ ]:
register_backbone(
    name="decoder",
    builder=lambda: DecoderOnlyTransformer(
        decoder_config
    ),
    hidden_dim=640,
    checkpoint_path=DECODER_CHECKPOINT,
)

register_backbone(
    name="ssm",
    builder=build_ssm_backbone,
    hidden_dim=SSM_DIM,
    checkpoint_path=SSM_CHECKPOINT,
)

print("Registered backbones:", list(BACKBONES))


# 8. Phase-2 task wrapper

Keep the Phase-1 backbone separate from the task head:

`backbone → [B,L,D] → your NER head`


In [ ]:
class Phase2NERModel(nn.Module):
    def __init__(
        self,
        backbone,
        hidden_dim,
        num_labels,
        num_start_classes,
        num_end_classes,
        dropout=0.1,
        boundary_loss_weight=0.3,
        bio_weights=None,
        start_weights=None,
        end_weights=None,
    ):
        super().__init__()

        self.backbone = backbone
        self.dropout = nn.Dropout(dropout)

        self.classifier = nn.Linear(
            hidden_dim,
            num_labels
        )

        self.start_classifier = nn.Linear(
            hidden_dim,
            num_start_classes
        )

        self.end_classifier = nn.Linear(
            hidden_dim,
            num_end_classes
        )

        self.boundary_loss_weight = boundary_loss_weight

        if bio_weights is not None:
            self.register_buffer(
                "bio_weights",
                bio_weights
            )
        else:
            self.bio_weights = None

        if start_weights is not None:
            self.register_buffer(
                "start_weights",
                start_weights
            )
        else:
            self.start_weights = None

        if end_weights is not None:
            self.register_buffer(
                "end_weights",
                end_weights
            )
        else:
            self.end_weights = None


    def forward(
        self,
        input_ids,
        attention_mask=None,
        labels=None,
        start_labels=None,
        end_labels=None,
    ):
        hidden = self.backbone.encode(
            input_ids,
            attention_mask=attention_mask
        )

        hidden = self.dropout(hidden)

        logits = self.classifier(hidden)
        start_logits = self.start_classifier(hidden)
        end_logits = self.end_classifier(hidden)

        loss = None
        bio_loss = None
        start_loss = None
        end_loss = None

        if labels is not None:
            bio_loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                labels.reshape(-1),
                weight=self.bio_weights,
                ignore_index=-100,
            )

            loss = bio_loss

            if start_labels is not None:
                start_loss = F.cross_entropy(
                    start_logits.reshape(
                        -1,
                        start_logits.size(-1)
                    ),
                    start_labels.reshape(-1),
                    weight=self.start_weights,
                    ignore_index=-100,
                )

                loss = (
                    loss
                    + self.boundary_loss_weight
                    * start_loss
                )

            if end_labels is not None:
                end_loss = F.cross_entropy(
                    end_logits.reshape(
                        -1,
                        end_logits.size(-1)
                    ),
                    end_labels.reshape(-1),
                    weight=self.end_weights,
                    ignore_index=-100,
                )

                loss = (
                    loss
                    + self.boundary_loss_weight
                    * end_loss
                )

        return {
            "loss": loss,
            "bio_loss": bio_loss,
            "start_loss": start_loss,
            "end_loss": end_loss,

            "logits": logits,
            "start_logits": start_logits,
            "end_logits": end_logits,
        }

In [ ]:
def build_phase2_model(
    backbone_name,
    num_labels,
    device,
    checkpoint_path=None,
    dropout=0.1,
    bio_weights=None,
    start_weights=None,
    end_weights=None,
):
    backbone = load_backbone(
        backbone_name,
        checkpoint_path=checkpoint_path,
        device=device,
    )

    hidden_dim = BACKBONES[
        backbone_name
    ]["hidden_dim"]

    model = Phase2NERModel(
        backbone=backbone,
        hidden_dim=hidden_dim,
        num_labels=num_labels,
        num_start_classes=len(START_DELTAS),
        num_end_classes=len(END_DELTAS),
        dropout=dropout,

        boundary_loss_weight=0.3,

        bio_weights=bio_weights,
        start_weights=start_weights,
        end_weights=end_weights,
    )

    return model.to(device)

# 9. Adaptation methods

Keep adaptation code separate so it can be reused in Phases 3–7.

Phase 2 requires meaningfully different adaptation experiments, including compulsory LORMA and a knowledge-distillation experiment.

Implement the substantive methods yourself from course material/papers.


In [ ]:
class LowRankLinear(nn.Module):
    def __init__(self, base_layer, rank=16, alpha=32.0):
        super().__init__()

        if not isinstance(base_layer, nn.Linear):
            raise TypeError("base_layer must be nn.Linear")

        self.base = base_layer

        for p in self.base.parameters():
            p.requires_grad = False

        self.rank = rank
        self.alpha = alpha
        self.scale = alpha / rank

        in_dim = base_layer.in_features
        out_dim = base_layer.out_features

        device = base_layer.weight.device
        dtype = base_layer.weight.dtype

        self.A = nn.Parameter(
            torch.empty(
                rank,
                in_dim,
                device=device,
                dtype=dtype,
            )
        )

        self.B = nn.Parameter(
            torch.zeros(
                out_dim,
                rank,
                device=device,
                dtype=dtype,
            )
        )

        nn.init.kaiming_uniform_(
            self.A,
            a=math.sqrt(5)
        )

    def forward(self, x):
        base_out = self.base(x)

        update = F.linear(
            F.linear(x, self.A),
            self.B
        )

        return base_out + self.scale * update

In [ ]:
ADAPTATION_METHODS = {}

def register_adaptation(name):
    def deco(fn):
        ADAPTATION_METHODS[name] = fn
        return fn
    return deco

@register_adaptation("full")
def apply_full_finetuning(model):
    for p in model.parameters():
        p.requires_grad = True
    return model

@register_adaptation("partial")
def apply_partial_finetuning(model):
    # Freeze whole backbone
    for p in model.backbone.parameters():
        p.requires_grad = False

    # Unfreeze last 2 transformer blocks
    if hasattr(model.backbone, "blocks"):
        blocks = model.backbone.blocks

    else:
        raise AttributeError(
            "partial fine-tuning needs a backbone with `blocks` "
            "(decoder only)."
        )

    for block in blocks[-2:]:
        for p in block.parameters():
            p.requires_grad = True

    # Unfreeze final normalization if present
    for name in [
        "ln_f",
        "final_ln",
        "final_norm",
        "norm",
    ]:
        if hasattr(model.backbone, name):
            for p in getattr(
                model.backbone,
                name
            ).parameters():
                p.requires_grad = True

            print("Unfroze final norm:", name)
            break

    # Always train the three NER heads
    for module in [
        model.classifier,
        model.start_classifier,
        model.end_classifier,
    ]:
        for p in module.parameters():
            p.requires_grad = True

    return model

# Note: "partial" and "lorma" target the decoder's transformer blocks;
# the SSM backbone is used with "full" fine-tuning only.
@register_adaptation("lorma")
def apply_lorma(model, rank=8, alpha=16.0):

    # freeze complete backbone
    for p in model.backbone.parameters():
        p.requires_grad = False

    # add low-rank trainable updates
    for block in model.backbone.blocks:

        block.attn.qkv = LowRankLinear(
            block.attn.qkv,
            rank=rank,
            alpha=alpha,
        )

        block.attn.out_proj = LowRankLinear(
            block.attn.out_proj,
            rank=rank,
            alpha=alpha,
        )

        block.mlp.fc_in = LowRankLinear(
            block.mlp.fc_in,
            rank=rank,
            alpha=alpha,
        )

        block.mlp.fc_out = LowRankLinear(
            block.mlp.fc_out,
            rank=rank,
            alpha=alpha,
        )

    for p in model.backbone.final_ln.parameters():
        p.requires_grad = True
    
    for module in [
        model.classifier,
        model.start_classifier,
        model.end_classifier,
    ]:
        for p in module.parameters():
            p.requires_grad = True

    return model



# 10. Experiment configuration

Keep every experiment reproducible. The active `exp_cfg` is the one trained below; the commented configurations are the other runs.


In [ ]:
from dataclasses import dataclass


@dataclass
class ExperimentConfig:
    backbone_name: str
    adaptation_method: str

    learning_rate: float
    weight_decay: float

    batch_size: int
    grad_accum: int
    num_epochs: int

    max_grad_norm: float = 1.0
    use_amp: bool = True

exp_cfg = ExperimentConfig(
    backbone_name="decoder",
    adaptation_method="full",

    learning_rate=5e-5,
    weight_decay=0.01,

    batch_size=2,
    grad_accum=2,
    num_epochs=8,
    max_grad_norm=1.0,
    use_amp=True,
)
# ---- Other runs (uncomment one to reproduce it) ----
# exp_cfg = ExperimentConfig(
#     backbone_name="decoder",
#     adaptation_method="partial",

#     learning_rate=3e-3,
#     weight_decay=0.01,

#     batch_size=2,
#     grad_accum=2,
#     num_epochs=8,
#     max_grad_norm=1.0,
#     use_amp=True,
# )
# exp_cfg = ExperimentConfig(
#     backbone_name="decoder",
#     adaptation_method="lorma",
#     learning_rate=5e-4,
#     weight_decay=0.01,
#     batch_size=2,
#     grad_accum=2,
#     num_epochs=10,
#     max_grad_norm=1.0,
#     use_amp=True,
# )

# SSM backbone: full fine-tuning only
# exp_cfg = ExperimentConfig(
#     backbone_name="ssm",
#     adaptation_method="full",
#     learning_rate=5e-5,
#     weight_decay=0.01,
#     batch_size=1,
#     grad_accum=4,
#     num_epochs=8,
#     max_grad_norm=1.0,
#     use_amp=True,
# )
# print(asdict(exp_cfg))


# 11. Dataset and collator


In [ ]:
train_dataset_ner = NERWindowDataset(
    train_windows_split
)

val_dataset_ner = NERWindowDataset(
    val_windows_split
)

train_loader = DataLoader(
    train_dataset_ner,
    batch_size=exp_cfg.batch_size,
    shuffle=True,
    collate_fn=ner_collator,
    pin_memory=torch.cuda.is_available(),
)

val_loader = DataLoader(
    val_dataset_ner,
    batch_size=exp_cfg.batch_size,
    shuffle=False,
    collate_fn=ner_collator,
    pin_memory=torch.cuda.is_available(),
)

# 12. Tiny overfit test

Before a full run, train on a tiny number of windows and verify that loss can decrease strongly. This catches broken labels, masking, padding, and checkpoint loading before you spend GPU time.


In [ ]:
def train_one_epoch(
    model,
    loader,
    optimizer,
    device,
    grad_accum=1,
    max_grad_norm=1.0,
    use_amp=True,
):
    model.train()

    total_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(loader):

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        labels = batch["labels"].to(device)
        start_labels = batch["start_labels"].to(device)
        end_labels = batch["end_labels"].to(device)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=use_amp and torch.cuda.is_available(),
        ):
            out = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
                start_labels=start_labels,
                end_labels=end_labels,
            )

            loss = out["loss"] / grad_accum

        loss.backward()

        should_step = (
            (step + 1) % grad_accum == 0
            or (step + 1) == len(loader)
        )

        if should_step:

            if max_grad_norm is not None:
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_grad_norm,
                )

            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        total_loss += loss.item() * grad_accum

    return total_loss / len(loader)

In [ ]:
from torch.utils.data import Subset, DataLoader

tiny_dataset = Subset(
    train_dataset_ner,
    range(min(4, len(train_dataset_ner)))
)

tiny_loader = DataLoader(
    tiny_dataset,
    batch_size=2,
    shuffle=True,
    collate_fn=ner_collator,
    pin_memory=torch.cuda.is_available(),
)

print("Tiny samples:", len(tiny_dataset))
print("Tiny batches :", len(tiny_loader))

Tiny samples: 4
Tiny batches : 2


In [ ]:
@torch.no_grad()
def run_validation(
    model,
    loader,
    device,
    use_amp=True,
):
    model.eval()

    total_loss = 0.0
    num_batches = 0

    all_bio_logits = []
    all_start_logits = []
    all_end_logits = []

    all_labels = []
    all_start_labels = []
    all_end_labels = []

    all_doc_ids = []
    all_window_ids = []

    all_token_starts = []
    all_token_ends = []

    all_token_offsets = []

    for batch in loader:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        labels = batch["labels"].to(device)
        start_labels = batch["start_labels"].to(device)
        end_labels = batch["end_labels"].to(device)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=use_amp and torch.cuda.is_available(),
        ):
            out = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
                start_labels=start_labels,
                end_labels=end_labels,
            )

        total_loss += out["loss"].item()
        num_batches += 1

        all_bio_logits.append(
            out["logits"].detach().cpu()
        )

        all_start_logits.append(
            out["start_logits"].detach().cpu()
        )

        all_end_logits.append(
            out["end_logits"].detach().cpu()
        )

        all_labels.append(
            labels.detach().cpu()
        )

        all_start_labels.append(
            start_labels.detach().cpu()
        )

        all_end_labels.append(
            end_labels.detach().cpu()
        )

        all_doc_ids.extend(
            batch["doc_id"]
        )

        all_window_ids.extend(
            batch["window_id"]
        )

        all_token_starts.extend(
            batch["token_start"]
        )

        all_token_ends.extend(
            batch["token_end"]
        )

        all_token_offsets.extend(
            batch["token_offsets"]
        )

    return {
        "loss": total_loss / max(num_batches, 1),

        "bio_logits": all_bio_logits,
        "start_logits": all_start_logits,
        "end_logits": all_end_logits,

        "labels": all_labels,
        "start_labels": all_start_labels,
        "end_labels": all_end_labels,

        "doc_ids": all_doc_ids,
        "window_ids": all_window_ids,

        "token_starts": all_token_starts,
        "token_ends": all_token_ends,

        "token_offsets": all_token_offsets,
    }

# 13. Decode predictions back to exact character spans

Use saved original `token_offsets`. Never reconstruct positions from normalized decoded text.


In [ ]:
def decode_window_predictions(
    bio_output,
    start_output,
    end_output,
    token_offsets,
):
    # -----------------------------------
    # BIO
    # -----------------------------------
    if bio_output.ndim == 2:
        bio_prob = torch.softmax(
            bio_output.float(),
            dim=-1
        )
        bio_ids = bio_prob.argmax(dim=-1).tolist()

    else:
        bio_ids = bio_output.tolist()
        bio_prob = None

    # -----------------------------------
    # START
    # -----------------------------------
    if start_output.ndim == 2:
        start_prob = torch.softmax(
            start_output.float(),
            dim=-1
        )
        start_ids = start_prob.argmax(dim=-1).tolist()

    else:
        start_ids = start_output.tolist()
        start_prob = None

    # -----------------------------------
    # END
    # -----------------------------------
    if end_output.ndim == 2:
        end_prob = torch.softmax(
            end_output.float(),
            dim=-1
        )
        end_ids = end_prob.argmax(dim=-1).tolist()

    else:
        end_ids = end_output.tolist()
        end_prob = None

    n = len(token_offsets)

    bio_ids = bio_ids[:n]
    start_ids = start_ids[:n]
    end_ids = end_ids[:n]

    spans = []

    cur_label = None
    first_token = None
    last_token = None

    def close_entity():
        nonlocal cur_label, first_token, last_token

        if cur_label is None:
            return

        token_start_char = token_offsets[first_token][0]
        token_end_char = token_offsets[last_token][1]

        sid = int(start_ids[first_token])
        eid = int(end_ids[last_token])

        start_delta = start_id2delta[sid]
        end_delta = end_id2delta[eid]

        char_start = token_start_char + start_delta
        char_end = token_end_char - end_delta

        if char_start < char_end:

            # Model logits case -> real confidence
            if bio_prob is not None:

                bio_conf = []

                for k in range(first_token, last_token + 1):
                    bid = int(bio_ids[k])
                    bio_conf.append(
                        bio_prob[k, bid].item()
                    )

                bio_conf = sum(bio_conf) / len(bio_conf)

                start_conf = start_prob[
                    first_token, sid
                ].item()

                end_conf = end_prob[
                    last_token, eid
                ].item()

                score = (
                    bio_conf
                    * start_conf
                    * end_conf
                ) ** (1 / 3)

            # Gold-ID sanity-test case
            else:
                score = 1.0

            spans.append([
                char_start,
                char_end,
                cur_label,
                score,
            ])

        cur_label = None
        first_token = None
        last_token = None

    for i in range(n):

        tag = id2label[int(bio_ids[i])]

        if tag == "O":
            close_entity()
            continue

        prefix, label = tag.split("-", 1)

        if prefix == "B":
            close_entity()

            cur_label = label
            first_token = i
            last_token = i

        elif prefix == "I":

            if cur_label == label:
                last_token = i

            else:
                close_entity()

                cur_label = label
                first_token = i
                last_token = i

    close_entity()

    return spans

In [ ]:
def exact_span_debug(
    text,
    gold_spans,
    pred_spans,
    limit=20,
):
    gold_set = {
        (int(s), int(e), lab)
        for s, e, lab in gold_spans
    }

    pred_set = {
        (int(s), int(e), lab)
        for s, e, lab in pred_spans
    }

    exact = gold_set & pred_set
    missed = gold_set - pred_set
    extra = pred_set - gold_set

    print("Exact matches:", len(exact))
    print("Missed:", len(missed))
    print("Extra :", len(extra))

    print("\nMISSED")

    for s, e, lab in sorted(missed)[:limit]:
        print(
            (s, e, lab),
            repr(text[s:e])
        )

    print("\nEXTRA")

    for s, e, lab in sorted(extra)[:limit]:
        print(
            (s, e, lab),
            repr(text[s:e])
        )

    return {
        "exact": exact,
        "missed": missed,
        "extra": extra,
    }

In [ ]:
bio_targets, start_targets, end_targets = build_token_targets(
    text,
    offsets,
    gold_spans
)

decoded = decode_window_predictions(
    bio_output=torch.tensor(bio_targets),
    start_output=torch.tensor(start_targets),
    end_output=torch.tensor(end_targets),
    token_offsets=offsets,
)
decoded_gold = [
    [s, e, label]
    for s, e, label, score in decoded
]


exact_span_debug(
    text,
    gold_spans,
    decoded_gold,
    limit=20
)

Exact matches: 104
Missed: 0
Extra : 0

MISSED

EXTRA


{'exact': {(137, 153, 'RESP'),
  (252, 261, 'DATE'),
  (276, 308, 'COURT'),
  (359, 369, 'COURT'),
  (575, 585, 'DATE'),
  (637, 644, 'WIT'),
  (695, 704, 'WIT'),
  (792, 800, 'APP'),
  (1122, 1130, 'APP'),
  (1341, 1349, 'APP'),
  (1373, 1380, 'WIT'),
  (1583, 1590, 'WIT'),
  (1809, 1822, 'WIT'),
  (2229, 2242, 'WIT'),
  (2296, 2303, 'WIT'),
  (2314, 2323, 'WIT'),
  (2341, 2348, 'WIT'),
  (2491, 2494, 'STAT'),
  (2501, 2513, 'WIT'),
  (2685, 2688, 'STAT'),
  (2756, 2763, 'WIT'),
  (2769, 2778, 'WIT'),
  (2791, 2801, 'WIT'),
  (2814, 2829, 'WIT'),
  (2941, 2948, 'STAT'),
  (3015, 3023, 'APP'),
  (3081, 3088, 'WIT'),
  (3102, 3111, 'WIT'),
  (3225, 3232, 'WIT'),
  (3583, 3591, 'APP'),
  (3707, 3716, 'DATE'),
  (3778, 3781, 'STAT'),
  (4013, 4045, 'COURT'),
  (4051, 4061, 'COURT'),
  (4091, 4100, 'DATE'),
  (4160, 4168, 'APP'),
  (4193, 4196, 'STAT'),
  (4319, 4322, 'STAT'),
  (4425, 4435, 'COURT'),
  (4464, 4467, 'STAT'),
  (4528, 4531, 'STAT'),
  (4651, 4661, 'COURT'),
  (4730, 4733, '

In [ ]:
from collections import defaultdict


def decode_validation_output(val_output):
    doc_predictions = defaultdict(list)

    meta_idx = 0

    for batch_idx in range(len(val_output["bio_logits"])):

        bio_batch = val_output["bio_logits"][batch_idx]
        start_batch = val_output["start_logits"][batch_idx]
        end_batch = val_output["end_logits"][batch_idx]

        batch_size = bio_batch.size(0)

        for j in range(batch_size):

            doc_id = val_output["doc_ids"][meta_idx]
            offsets = val_output["token_offsets"][meta_idx]

            # Important: create document even if no spans predicted
            _ = doc_predictions[doc_id]

            n = len(offsets)

            spans = decode_window_predictions(
                bio_output=bio_batch[j, :n],
                start_output=start_batch[j, :n],
                end_output=end_batch[j, :n],
                token_offsets=offsets,
            )

            doc_predictions[doc_id].extend(spans)

            meta_idx += 1

    return dict(doc_predictions)

In [ ]:
def merge_document_predictions(doc_predictions):
    merged = {}

    for doc_id, spans in doc_predictions.items():

        best = {}

        for start, end, label, score in spans:
            key = (
                int(start),
                int(end),
                label
            )

            if (
                key not in best
                or score > best[key]
            ):
                best[key] = score

        merged[doc_id] = [
            [s, e, label, score]
            for (s, e, label), score
            in sorted(
                best.items(),
                key=lambda x: (
                    x[0][0],
                    x[0][1],
                    x[0][2],
                )
            )
        ]

    return merged

In [ ]:
def suppress_near_duplicates(predictions, overlap_thr=0.8):
    out = {}

    for doc_id, spans in predictions.items():
        spans = sorted(
            spans,
            key=lambda x: x[3],
            reverse=True
        )

        kept = []

        for cur in spans:
            s1, e1, l1, sc1 = cur

            duplicate = False

            for old in kept:
                s2, e2, l2, sc2 = old

                if l1 != l2:
                    continue

                inter = max(
                    0,
                    min(e1, e2) - max(s1, s2)
                )

                union = max(e1, e2) - min(s1, s2)

                iou = inter / union if union > 0 else 0

                if iou >= overlap_thr:
                    duplicate = True
                    break

            if not duplicate:
                kept.append(cur)

        out[doc_id] = kept

    return out

In [ ]:
def strict_span_metrics(
    predictions,
    gold_map,
    doc_ids,
    labels,
):
    per_label = {}
    macro_f1 = 0.0

    for label in labels:
        pred_set = set()
        gold_set = set()

        for doc_id in doc_ids:
            doc_id = str(doc_id)

            # predictions may be:
            # [start, end, label]
            # or [start, end, label, score]
            for row in predictions.get(doc_id, []):
                s, e, lab = row[:3]

                if lab == label:
                    pred_set.add(
                        (doc_id, int(s), int(e), lab)
                    )

            for row in normalize_gold(
                gold_map.get(doc_id, [])
            ):
                s, e, lab = row[:3]

                if lab == label:
                    gold_set.add(
                        (doc_id, int(s), int(e), lab)
                    )

        tp = len(pred_set & gold_set)
        fp = len(pred_set - gold_set)
        fn = len(gold_set - pred_set)

        precision = (
            tp / (tp + fp)
            if tp + fp > 0
            else 0.0
        )

        recall = (
            tp / (tp + fn)
            if tp + fn > 0
            else 0.0
        )

        f1 = (
            2 * precision * recall
            / (precision + recall)
            if precision + recall > 0
            else 0.0
        )

        per_label[label] = {
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "tp": tp,
            "fp": fp,
            "fn": fn,
        }

        macro_f1 += f1

    macro_f1 /= len(labels)

    return {
        "macro_f1": macro_f1,
        "per_label": per_label,
    }

##  Training

In [ ]:
##fresh model

BACKBONE_LR = 7e-5 #increase

HEAD_LR = 3e-4
#5e-4
if "ner_model" in globals():
    del ner_model

if torch.cuda.is_available():
    torch.cuda.empty_cache()

ner_model = build_phase2_model(
    backbone_name=exp_cfg.backbone_name,
    num_labels=len(label2id),
    device=DEVICE,

    bio_weights=bio_weights,
    start_weights=start_weights,
    end_weights=end_weights,
)

ner_model = ADAPTATION_METHODS[
    exp_cfg.adaptation_method
](ner_model)


# optimizer = torch.optim.AdamW(
#     [
#         p for p in ner_model.parameters()
#         if p.requires_grad
#     ],
#     lr=5e-5,
#     weight_decay=exp_cfg.weight_decay,
# )

optimizer = torch.optim.AdamW(
    [
        {
            "params": ner_model.backbone.parameters(),
            "lr": BACKBONE_LR,
        },
        {
            "params": ner_model.classifier.parameters(),
            "lr": HEAD_LR,
        },
        {
            "params": ner_model.start_classifier.parameters(),
            "lr": HEAD_LR,
        },
        {
            "params": ner_model.end_classifier.parameters(),
            "lr": HEAD_LR,
        },
    ],
    weight_decay=exp_cfg.weight_decay,
)
##checks
model_param_ids = {
    id(p)
    for p in ner_model.parameters()
    if p.requires_grad
}

optimizer_param_ids = {
    id(p)
    for group in optimizer.param_groups
    for p in group["params"]
}

print(
    "Optimizer matches model:",
    model_param_ids == optimizer_param_ids
)

In [ ]:
all_gold = {
    **fold1_gold,
    **fold2_gold,
}

In [ ]:
from pathlib import Path

# One output folder per experiment, e.g. phase2_decoder_full, phase2_decoder_lorma, phase2_ssm_full
RUN_DIR = Path(f"/kaggle/working/phase2_{exp_cfg.backbone_name}_{exp_cfg.adaptation_method}")
RUN_DIR.mkdir(parents=True, exist_ok=True)

best_f1 = -1.0

for epoch in range(exp_cfg.num_epochs):

    t0 = time.time()

    train_loss = train_one_epoch(
        model=ner_model,
        loader=train_loader,
        optimizer=optimizer,
        device=DEVICE,
        grad_accum=exp_cfg.grad_accum,
        max_grad_norm=exp_cfg.max_grad_norm,
        use_amp=exp_cfg.use_amp,
    )

    val_output = run_validation(
        model=ner_model,
        loader=val_loader,
        device=DEVICE,
        use_amp=exp_cfg.use_amp,
    )

    window_predictions = decode_validation_output(
        val_output
    )

    merged_predictions = merge_document_predictions(
        window_predictions
    )
    #remove later
    merged_predictions = suppress_near_duplicates(
    merged_predictions,
    overlap_thr=0.8
)
    metrics = strict_span_metrics(
        predictions=merged_predictions,
        gold_map=all_gold,
        doc_ids=val_doc_ids,
        labels=entity_labels,
    )

    f1 = metrics["macro_f1"]

    print(
        f"Epoch {epoch + 1:02d}/{exp_cfg.num_epochs} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_output['loss']:.4f} | "
        f"strict_macro_f1={f1:.4f} | "
        f"time={(time.time() - t0)/60:.1f} min"
    )

    if f1 > best_f1:
        best_f1 = f1

        torch.save(
            ner_model.state_dict(),
            RUN_DIR / "best_model.pt"
        )

        with open(
            RUN_DIR / "validation_predictions.json",
            "w",
            encoding="utf-8",
        ) as f:
            json.dump(
                merged_predictions,
                f,
                ensure_ascii=False,
                indent=2,
            )

        with open(
            RUN_DIR / "metrics.json",
            "w",
            encoding="utf-8",
        ) as f:
            json.dump(
                metrics,
                f,
                indent=2,
            )

        print(
            f"  -> saved new best model: {best_f1:.4f}"
        )

print("\nBEST STRICT MACRO-F1:", best_f1)

Epoch 01/8 | train_loss=1.0088 | val_loss=0.6116 | strict_macro_f1=0.2211 | time=0.7 min
  -> saved new best model: 0.2211
Epoch 02/8 | train_loss=0.3275 | val_loss=0.5820 | strict_macro_f1=0.2526 | time=0.6 min
  -> saved new best model: 0.2526
Epoch 03/8 | train_loss=0.2043 | val_loss=0.6147 | strict_macro_f1=0.2435 | time=0.6 min
Epoch 04/8 | train_loss=0.1281 | val_loss=0.7244 | strict_macro_f1=0.2672 | time=0.6 min
  -> saved new best model: 0.2672
Epoch 05/8 | train_loss=0.0970 | val_loss=0.7784 | strict_macro_f1=0.2739 | time=0.6 min
  -> saved new best model: 0.2739
Epoch 06/8 | train_loss=0.0807 | val_loss=0.8153 | strict_macro_f1=0.3127 | time=0.6 min
  -> saved new best model: 0.3127
Epoch 07/8 | train_loss=0.0595 | val_loss=1.0538 | strict_macro_f1=0.2899 | time=0.6 min
Epoch 08/8 | train_loss=0.0507 | val_loss=1.0521 | strict_macro_f1=0.3086 | time=0.6 min

BEST STRICT MACRO-F1: 0.31265201572774687


In [ ]:
#to check finetuning params
total = sum(
    p.numel()
    for p in ner_model.parameters()
)

trainable = sum(
    p.numel()
    for p in ner_model.parameters()
    if p.requires_grad
)

print("Total params    :", total)
print("Trainable params:", trainable)
print(
    "Trainable %     :",
    100 * trainable / total
)

In [ ]:
scores = []

for spans in merged_predictions.values():
    for s, e, label, score in spans:
        scores.append(score)

print("Number of spans:", len(scores))
print("Min:", min(scores))
print("Max:", max(scores))
print("Mean:", sum(scores) / len(scores))

scores_sorted = sorted(scores)

for q in [0.5, 0.75, 0.9, 0.95, 0.99]:
    idx = int(q * (len(scores_sorted) - 1))
    print(f"{q:.2f} quantile:", scores_sorted[idx])

In [ ]:
def apply_confidence_threshold(
    predictions,
    threshold,
):
    out = {}

    for doc_id, spans in predictions.items():

        out[doc_id] = [
            [s, e, label]
            for s, e, label, score in spans
            if score >= threshold
        ]

    return out



best_threshold = None
best_f1 = -1.0

for threshold in [
    0.0, 0.05, 0.10, 0.15, 0.20,
    0.25, 0.30, 0.35, 0.40, 0.45,
    0.50, 0.55, 0.60, 0.65, 0.70,
    0.75, 0.80, 0.85, 0.90,0.91,0.95,0.97
]:
    filtered = apply_confidence_threshold(
        merged_predictions,
        threshold
    )

    metrics = strict_span_metrics(
        predictions=filtered,
        gold_map=all_gold,
        doc_ids=val_doc_ids,
        labels=entity_labels,
    )

    f1 = metrics["macro_f1"]

    n_pred = sum(
        len(v)
        for v in filtered.values()
    )

    print(
        f"threshold={threshold:.2f} | "
        f"pred={n_pred} | "
        f"Macro-F1={f1:.4f}"
    )

    if f1 > best_f1:
        best_f1 = f1
        best_threshold = threshold

print(
    "\nBest threshold:",
    best_threshold,
    "| F1:",
    best_f1
)

threshold=0.00 | pred=2148 | Macro-F1=0.3086
threshold=0.05 | pred=2148 | Macro-F1=0.3086
threshold=0.10 | pred=2148 | Macro-F1=0.3086
threshold=0.15 | pred=2148 | Macro-F1=0.3086
threshold=0.20 | pred=2148 | Macro-F1=0.3086
threshold=0.25 | pred=2148 | Macro-F1=0.3086
threshold=0.30 | pred=2148 | Macro-F1=0.3086
threshold=0.35 | pred=2148 | Macro-F1=0.3086
threshold=0.40 | pred=2148 | Macro-F1=0.3086
threshold=0.45 | pred=2148 | Macro-F1=0.3086
threshold=0.50 | pred=2146 | Macro-F1=0.3089
threshold=0.55 | pred=2144 | Macro-F1=0.3092
threshold=0.60 | pred=2132 | Macro-F1=0.3104
threshold=0.65 | pred=2107 | Macro-F1=0.3133
threshold=0.70 | pred=2062 | Macro-F1=0.3172
threshold=0.75 | pred=1975 | Macro-F1=0.3234
threshold=0.80 | pred=1817 | Macro-F1=0.3407
threshold=0.85 | pred=1605 | Macro-F1=0.3612
threshold=0.90 | pred=1367 | Macro-F1=0.3755
threshold=0.91 | pred=1311 | Macro-F1=0.3800
threshold=0.95 | pred=1061 | Macro-F1=0.3953
threshold=0.97 | pred=910 | Macro-F1=0.3803

Best thres

In [ ]:
#multiple threshold
def apply_label_thresholds(predictions, thresholds):
    out = {}

    for doc_id, spans in predictions.items():
        out[doc_id] = []

        for s, e, label, score in spans:
            if score >= thresholds[label]:
                out[doc_id].append([s, e, label])

    return out

In [ ]:
label_thresholds = {}

for label in entity_labels:

    best_t = 0.0
    best_f1 = -1.0

    print("\nLABEL:", label)

    for i in range(50, 100):
        threshold = i / 100.0

        filtered = {}

        for doc_id, spans in merged_predictions.items():
            filtered[doc_id] = [
                [s, e, lab]
                for s, e, lab, score in spans
                if lab == label and score >= threshold
            ]

        metrics = strict_span_metrics(
            predictions=filtered,
            gold_map=all_gold,
            doc_ids=val_doc_ids,
            labels=[label],
        )

        f1 = metrics["macro_f1"]

        if f1 > best_f1:
            best_f1 = f1
            best_t = threshold

    label_thresholds[label] = best_t

    print(
        f"best threshold={best_t:.2f} | "
        f"F1={best_f1:.4f}"
    )

print("\nPer-label thresholds:")

for label, threshold in label_thresholds.items():
    print(
        f"{label:12s}: {threshold:.2f}"
    )


LABEL: APP
best threshold=0.97 | F1=0.1944

LABEL: RESP
best threshold=0.88 | F1=0.1412

LABEL: A.COUNSEL
best threshold=0.95 | F1=0.4324

LABEL: R.COUNSEL
best threshold=0.92 | F1=0.4151

LABEL: JUDGE
best threshold=0.92 | F1=0.5526

LABEL: WIT
best threshold=0.50 | F1=0.0000

LABEL: AUTH
best threshold=0.96 | F1=0.1319

LABEL: COURT
best threshold=0.99 | F1=0.8172

LABEL: STAT
best threshold=0.98 | F1=0.6686

LABEL: PREC
best threshold=0.98 | F1=0.3373

LABEL: DATE
best threshold=0.99 | F1=0.8825

LABEL: CASENO
best threshold=0.99 | F1=0.4900

Per-label thresholds:
APP         : 0.97
RESP        : 0.88
A.COUNSEL   : 0.95
R.COUNSEL   : 0.92
JUDGE       : 0.92
WIT         : 0.50
AUTH        : 0.96
COURT       : 0.99
STAT        : 0.98
PREC        : 0.98
DATE        : 0.99
CASENO      : 0.99


In [ ]:
filtered_predictions = apply_label_thresholds(
    merged_predictions,
    label_thresholds
)

metrics = strict_span_metrics(
    predictions=filtered_predictions,
    gold_map=all_gold,
    doc_ids=val_doc_ids,
    labels=entity_labels,
)

print(
    "Per-label threshold Macro-F1:",
    metrics["macro_f1"]
)

Per-label threshold Macro-F1: 0.4219346488315805


In [ ]:
CONFIDENCE_THRESHOLD = 0.96
# CONFIDENCE_THRESHOLD = 0.92 #lorma
# CONFIDENCE_THRESHOLD = 0.92 #ssm full



filtered_predictions = apply_confidence_threshold(
    merged_predictions,
    CONFIDENCE_THRESHOLD
)

metrics = strict_span_metrics(
    predictions=filtered_predictions,
    gold_map=all_gold,
    doc_ids=val_doc_ids,
    labels=entity_labels,
)

In [ ]:
raw_metrics = strict_span_metrics(
    predictions=merged_predictions,
    gold_map=all_gold,
    doc_ids=val_doc_ids,
    labels=entity_labels,
)

print(
    "Raw Macro-F1:",
    raw_metrics["macro_f1"]
)

In [ ]:
import shutil

# Zip this run's folder for download.
shutil.make_archive(
    str(RUN_DIR),
    "zip",
    str(RUN_DIR)
)


In [ ]:
# Per-label validation metrics of the best checkpoint of this run.
metrics_path = RUN_DIR / "metrics.json"

if metrics_path.exists():
    with open(metrics_path, "r") as f:
        m = json.load(f)

    print("Macro F1:", m["macro_f1"])

    for label, x in m["per_label"].items():
        print(
            f"{label:10s} "
            f"P={x['precision']:.3f} "
            f"R={x['recall']:.3f} "
            f"F1={x['f1']:.3f} "
            f"TP={x['tp']} "
            f"FP={x['fp']} "
            f"FN={x['fn']}"
        )
else:
    print("No metrics file yet:", metrics_path)


# 13b. Test windows and inference helper

Used by the knowledge-distillation inference and by the final submission.


In [ ]:
test_windows = []

for doc_id, text in test_text.items():
    test_windows.extend(
        make_windows(
            doc_id=doc_id,
            text=text,
            gold_spans=[],
            tokenizer=tokenizer,
            max_tokens=2048,
            stride=1536,
        )
    )

test_dataset_ner = NERWindowDataset(test_windows)

test_loader = DataLoader(
    test_dataset_ner,
    batch_size=exp_cfg.batch_size,
    shuffle=False,
    collate_fn=ner_collator,
    pin_memory=torch.cuda.is_available(),
)


@torch.no_grad()
def run_inference(model, loader, device, use_amp=True):
    model.eval()

    bio_logits = []
    start_logits = []
    end_logits = []

    doc_ids = []
    window_ids = []
    token_starts = []
    token_ends = []
    token_offsets = []

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=use_amp and torch.cuda.is_available(),
        ):
            out = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
            )

        bio_logits.append(out["logits"].detach().cpu())
        start_logits.append(out["start_logits"].detach().cpu())
        end_logits.append(out["end_logits"].detach().cpu())

        doc_ids.extend(batch["doc_id"])
        window_ids.extend(batch["window_id"])
        token_starts.extend(batch["token_start"])
        token_ends.extend(batch["token_end"])
        token_offsets.extend(batch["token_offsets"])

    return {
        "bio_logits": bio_logits,
        "start_logits": start_logits,
        "end_logits": end_logits,
        "doc_ids": doc_ids,
        "window_ids": window_ids,
        "token_starts": token_starts,
        "token_ends": token_ends,
        "token_offsets": token_offsets,
    }


print("Test docs   :", len(test_text))
print("Test windows:", len(test_windows))

Test docs   : 35
Test windows: 167


# 14. Knowledge distillation experiment

Keep this separate:

`teacher → teacher signals → smaller student`

Choose teacher/student pairing, signal, loss, temperature, and weighting yourself.


In [ ]:
from dataclasses import replace
import copy

TEACHER_CKPT = "/kaggle/working/phase2_decoder_full/best_model.pt"

# fresh 8-layer teacher
teacher = build_phase2_model(
    backbone_name="decoder",
    num_labels=len(label2id),
    device="cpu",
    bio_weights=bio_weights,
    start_weights=start_weights,
    end_weights=end_weights,
)

state = torch.load(
    TEACHER_CKPT,
    map_location="cpu"
)

teacher.load_state_dict(
    state,
    strict=True
)

teacher = teacher.to(DEVICE)
teacher.eval()

for p in teacher.parameters():
    p.requires_grad = False

print("Teacher loaded")

print(
    "Teacher params:",
    sum(p.numel() for p in teacher.parameters())
)

In [ ]:
# -----------------------------------------
# Smaller 6-layer decoder student
# -----------------------------------------

student_config = replace(
    decoder_config,
    n_layer=6
)

student_backbone = DecoderOnlyTransformer(
    student_config
)

# -----------------------------------------
# Copy embeddings
# -----------------------------------------

student_backbone.token_embedding.load_state_dict(
    teacher.backbone.token_embedding.state_dict()
)

student_backbone.position_embedding.load_state_dict(
    teacher.backbone.position_embedding.state_dict()
)

# -----------------------------------------
# Copy selected teacher blocks
# -----------------------------------------

# Student block i is initialised from teacher block layer_map[i];
# every student block is initialised (no randomly initialised layers).
layer_map = [0, 1, 3, 4, 6, 7]

assert len(layer_map) == student_config.n_layer, (
    len(layer_map), student_config.n_layer
)

for student_idx, teacher_idx in enumerate(layer_map):
    student_backbone.blocks[
        student_idx
    ].load_state_dict(
        teacher.backbone.blocks[
            teacher_idx
        ].state_dict()
    )

# final normalization
student_backbone.final_ln.load_state_dict(
    teacher.backbone.final_ln.state_dict()
)

# -----------------------------------------
# Wrap using same Phase-2 heads
# -----------------------------------------

student = Phase2NERModel(
    backbone=student_backbone,
    hidden_dim=decoder_config.n_embd,
    num_labels=len(label2id),

    num_start_classes=len(START_DELTAS),
    num_end_classes=len(END_DELTAS),

    dropout=0.1,
    boundary_loss_weight=0.3,

    bio_weights=bio_weights,
    start_weights=start_weights,
    end_weights=end_weights,
)

# Copy teacher NER heads as initialization
student.classifier.load_state_dict(
    teacher.classifier.state_dict()
)

student.start_classifier.load_state_dict(
    teacher.start_classifier.state_dict()
)

student.end_classifier.load_state_dict(
    teacher.end_classifier.state_dict()
)

student = student.to(DEVICE)

teacher_params = sum(
    p.numel() for p in teacher.parameters()
)

student_params = sum(
    p.numel() for p in student.parameters()
)

print("Teacher params:", teacher_params)
print("Student params:", student_params)

print(
    "Student size %:",
    100 * student_params / teacher_params
)

In [ ]:
class DistillationExperiment:
    def __init__(
        self,
        teacher,
        student,
        temperature=2.0,
        alpha=0.5,
    ):
        self.teacher = teacher
        self.student = student

        self.temperature = temperature
        self.alpha = alpha

        self.teacher.eval()

        for p in self.teacher.parameters():
            p.requires_grad = False


    def masked_kd(
        self,
        student_logits,
        teacher_logits,
        mask,
    ):
        T = self.temperature

        s_log = F.log_softmax(
            student_logits / T,
            dim=-1
        )

        t_prob = F.softmax(
            teacher_logits / T,
            dim=-1
        )

        # KL per token
        kl = F.kl_div(
            s_log,
            t_prob,
            reduction="none"
        ).sum(dim=-1)

        mask = mask.float()

        loss = (
            (kl * mask).sum()
            / mask.sum().clamp_min(1.0)
        )

        return loss * (T * T)


    def compute_loss(
        self,
        batch,
        device,
    ):
        input_ids = batch[
            "input_ids"
        ].to(device)

        attention_mask = batch[
            "attention_mask"
        ].to(device)

        labels = batch[
            "labels"
        ].to(device)

        start_labels = batch[
            "start_labels"
        ].to(device)

        end_labels = batch[
            "end_labels"
        ].to(device)

        # -----------------------
        # Teacher
        # -----------------------

        with torch.no_grad():
            teacher_out = self.teacher(
                input_ids=input_ids,
                attention_mask=attention_mask,
            )

        # -----------------------
        # Student
        # -----------------------

        student_out = self.student(
            input_ids=input_ids,
            attention_mask=attention_mask,

            labels=labels,
            start_labels=start_labels,
            end_labels=end_labels,
        )

        gold_loss = student_out["loss"]

        # all non-padding tokens
        bio_mask = attention_mask.bool()

        # only entity-boundary tokens
        start_mask = start_labels != -100
        end_mask = end_labels != -100

        bio_kd = self.masked_kd(
            student_out["logits"],
            teacher_out["logits"],
            bio_mask,
        )

        start_kd = self.masked_kd(
            student_out["start_logits"],
            teacher_out["start_logits"],
            start_mask,
        )

        end_kd = self.masked_kd(
            student_out["end_logits"],
            teacher_out["end_logits"],
            end_mask,
        )

        kd_loss = (
            bio_kd
            + 0.1 * start_kd
            + 0.1 * end_kd
        )

        loss = (
            (1.0 - self.alpha) * gold_loss
            + self.alpha * kd_loss
        )

        return {
            "loss": loss,
            "gold_loss": gold_loss,
            "kd_loss": kd_loss,
        }

In [ ]:
kd_exp = DistillationExperiment(
    teacher=teacher,
    student=student,
    temperature=2.0,
    alpha=0.15,
)

KD_LR = 3e-4
KD_EPOCHS = 8

optimizer = torch.optim.AdamW(
    [
        p for p in student.parameters()
        if p.requires_grad
    ],
    lr=KD_LR,
    weight_decay=0.01,
)

In [ ]:
def train_kd_epoch(
    kd_exp,
    loader,
    optimizer,
    device,
    grad_accum=2,
    max_grad_norm=1.0,
):
    kd_exp.student.train()
    kd_exp.teacher.eval()

    optimizer.zero_grad(set_to_none=True)

    total = 0.0
    total_gold = 0.0
    total_kd = 0.0

    for step, batch in enumerate(loader):

        out = kd_exp.compute_loss(
            batch,
            device
        )

        loss = out["loss"] / grad_accum

        loss.backward()

        should_step = (
            (step + 1) % grad_accum == 0
            or
            (step + 1) == len(loader)
        )

        if should_step:

            torch.nn.utils.clip_grad_norm_(
                kd_exp.student.parameters(),
                max_grad_norm
            )

            optimizer.step()
            optimizer.zero_grad(
                set_to_none=True
            )

        total += out["loss"].item()
        total_gold += out[
            "gold_loss"
        ].item()

        total_kd += out[
            "kd_loss"
        ].item()

    n = len(loader)

    return (
        total / n,
        total_gold / n,
        total_kd / n,
    )

In [ ]:
KD_RUN_DIR = Path(
    "/kaggle/working/phase2_decoder_kd"
)

KD_RUN_DIR.mkdir(
    parents=True,
    exist_ok=True
)

best_f1 = -1.0

for epoch in range(KD_EPOCHS):

    t0 = time.time()

    train_loss, gold_loss, kd_loss = train_kd_epoch(
        kd_exp=kd_exp,
        loader=train_loader,
        optimizer=optimizer,
        device=DEVICE,
        grad_accum=2,
    )

    val_output = run_validation(
        model=student,
        loader=val_loader,
        device=DEVICE,
        use_amp=exp_cfg.use_amp,
    )

    window_predictions = decode_validation_output(
        val_output
    )

    merged_predictions = merge_document_predictions(
        window_predictions
    )

    metrics = strict_span_metrics(
        predictions=merged_predictions,
        gold_map=all_gold,
        doc_ids=val_doc_ids,
        labels=entity_labels,
    )

    f1 = metrics["macro_f1"]

    print(
        f"Epoch {epoch+1:02d}/{KD_EPOCHS} | "
        f"loss={train_loss:.4f} | "
        f"gold={gold_loss:.4f} | "
        f"kd={kd_loss:.4f} | "
        f"val={val_output['loss']:.4f} | "
        f"F1={f1:.4f} | "
        f"{(time.time()-t0)/60:.1f} min"
    )

    if f1 > best_f1:
        best_f1 = f1

        torch.save(
            student.state_dict(),
            KD_RUN_DIR / "best_model.pt"
        )

        print(
            f"  -> NEW BEST: {best_f1:.4f}"
        )

print(
    "\nBEST KD F1:",
    best_f1
)

In [ ]:
student.load_state_dict(
    torch.load(
        "/kaggle/working/phase2_decoder_kd/best_model.pt",
        map_location=DEVICE
    )
)

student = student.to(DEVICE)
student.eval()

In [ ]:
test_output = run_inference(
    model=student,
    loader=test_loader,
    device=DEVICE,
    use_amp=True,
)

window_predictions = decode_validation_output(
    test_output
)

merged_predictions = merge_document_predictions(
    window_predictions
)

KD_THRESHOLD = 0.90

final_predictions = apply_confidence_threshold(
    merged_predictions,
    KD_THRESHOLD
)

# KD predictions are kept in the KD run folder; the main submission is written in section 15.
pred_path = KD_RUN_DIR / "prediction.json"

with open(pred_path, "w") as f:
    json.dump(
        final_predictions,
        f,
        ensure_ascii=False
    )

print("Saved:", pred_path)

# 15. Submission validation

Expected Phase-2 output:

```json
{
  "1128835": [
    [18, 40, "COURT"],
    [114, 162, "CASENO"]
  ]
}
```


In [ ]:
BEST_MODEL_PATH = RUN_DIR / "best_model.pt"

state = torch.load(
    BEST_MODEL_PATH,
    map_location=DEVICE
)

ner_model.load_state_dict(state, strict=True)
ner_model.to(DEVICE)
ner_model.eval()

test_output = run_inference(
    ner_model,
    test_loader,
    DEVICE,
    use_amp=exp_cfg.use_amp,
)

test_window_predictions = decode_validation_output(
    test_output
)

test_merged_predictions = merge_document_predictions(
    test_window_predictions
)


# "global"   : single CONFIDENCE_THRESHOLD (used for the best submission)
# "per_label": thresholds tuned per entity type (section 13)
THRESHOLD_MODE = "global"

if THRESHOLD_MODE == "per_label":
    test_predictions = apply_label_thresholds(
        test_merged_predictions,
        label_thresholds
    )
else:
    test_predictions = apply_confidence_threshold(
        test_merged_predictions,
        CONFIDENCE_THRESHOLD
    )

for doc_id in test_text:
    test_predictions.setdefault(
        str(doc_id),
        []
    )

test_predictions = {
    str(doc_id): spans
    for doc_id, spans in test_predictions.items()
}

print(
    "Predicted entities:",
    sum(len(x) for x in test_predictions.values())
)

In [ ]:
def validate_prediction_json(predictions, test_documents, valid_labels):
    expected_ids = set(map(str, test_documents.keys()))
    got_ids = set(map(str, predictions.keys()))

    missing = expected_ids - got_ids
    extra = got_ids - expected_ids

    assert not missing, f"Missing IDs: {list(missing)[:10]}"
    assert not extra, f"Unknown IDs: {list(extra)[:10]}"

    valid_labels = set(valid_labels)

    for doc_id, spans in predictions.items():
        text = test_documents[str(doc_id)]
        assert isinstance(spans, list)

        for row in spans:
            assert len(row) in (3, 4), (doc_id, row)
            start, end, label = row[:3]

            assert isinstance(start, int)
            assert isinstance(end, int)
            assert 0 <= start <= end <= len(text), (doc_id, row, len(text))
            assert label in valid_labels, (doc_id, row)

    print("prediction.json structure looks valid.")


In [ ]:
validate_prediction_json(
    predictions=test_predictions,
    test_documents=test_text,
    valid_labels=entity_labels,
)

OUTPUT_PATH = "/kaggle/working/prediction.json"

with open(
    OUTPUT_PATH,
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        test_predictions,
        f,
        ensure_ascii=False,
        indent=2,
    )

print("Saved:", OUTPUT_PATH)
print("Documents:", len(test_predictions))

prediction.json structure looks valid.
Saved: /kaggle/working/prediction.json
Documents: 35


In [ ]:
def save_submission(predictions, output_dir):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    pred_path = output_dir / "prediction.json"
    zip_path = output_dir / "submission.zip"

    with open(pred_path, "w", encoding="utf-8") as f:
        json.dump(predictions, f, ensure_ascii=False, indent=2)

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        z.write(pred_path, arcname="prediction.json")

    print("Saved:", pred_path)
    print("Saved:", zip_path)

    return pred_path, zip_path

save_submission(test_predictions,"/kaggle/working")

Saved: /kaggle/working/prediction.json
Saved: /kaggle/working/submission.zip


(PosixPath('/kaggle/working/prediction.json'),
 PosixPath('/kaggle/working/submission.zip'))

# 16. Experiment bookkeeping

Each run writes to `/kaggle/working/phase2_<backbone>_<method>/` (see `RUN_DIR`), e.g.:

```text
phase2_decoder_full/      best_model.pt, metrics.json, validation_predictions.json
phase2_decoder_partial/
phase2_decoder_lorma/
phase2_ssm_full/
phase2_decoder_kd/        best_model.pt, prediction.json
```


In [ ]:
def save_experiment_config(cfg, run_dir):
    run_dir = Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)

    with open(run_dir / "config.json", "w", encoding="utf-8") as f:
        json.dump(asdict(cfg), f, indent=2)


# 17. Pre-submission checklist

- [ ] Exact Phase-1 checkpoint loaded
- [ ] No external pretrained weights
- [ ] Original text not normalized before offset calculation
- [ ] `encode_with_offsets()` manually verified
- [ ] Gold span convention verified
- [ ] Window predictions map to original character coordinates
- [ ] Overlap duplicates handled
- [ ] Every test ID present
- [ ] No unknown IDs
- [ ] Only official labels used
- [ ] All spans lie inside the document
- [ ] LORMA experiment included
- [ ] Knowledge-distillation experiment included
- [ ] `prediction.json` is at the root of `submission.zip`
- [ ] Config/checkpoint/predictions/results saved per experiment
